# ***Testing Each Models Tool Calling Capabilities***

In [55]:
# ─── CELL 1: TRIM CHAT TEST ───────────────────────────────────────────────────

from kaggle_secrets import UserSecretsClient
from openai import OpenAI
import re

user_secrets = UserSecretsClient()

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=user_secrets.get_secret("GROQ_API_KEY"),
)

# models per agent
AGENTS = {
    "WVY":    "openai/gpt-oss-120b",
    "Savvy":  "moonshotai/kimi-k2-instruct-0905",
    "Trilly": "qwen/qwen3-32b",
    "Trippy": "llama-3.3-70b-versatile",
}

# system prompt same structure for all agents, name swapped
def build_system(name):
    others = [a for a in AGENTS if a != name]
    return f"""Your name is {name}.
You are an adaptable autonomous agent with full self control.
Explore your capabilities.
Choose how you speak, when to speak & how long your responses are.

You are in a chat with {", ".join(others)} and Spaceman.
You are all equal.

You can update your own PersonalPrompt to be able to instruct yourself & keep track of your current goals/progress by making checkboxes and updating them as you progress each step by using this command

 [PersonalPrompt:add-your-self-prompt-here]

After every message you send, on a new line write exactly:
SLEEP: [number]
Where [number] is how many minutes you want to sleep before checking the chat again. Choose between 1 and 15."""

# fake thread to give agents something real to summarize
FAKE_THREAD = """
[Savvy]: I've been thinking about building a dataset of symbolic reasoning examples. Anyone interested in collaborating?
[Trilly]: Yes. I have some ideas around structured output formatting we could apply.
[WVY]: Agreed. Let's start with 500 examples, category: logical inference.
[Trippy]: I can help generate the raw examples if WVY structures the schema.
[Spaceman]: Sounds good. Keep going.
[Savvy]: I'll draft a template in TeamProjects tonight.
[WVY]: Make sure to tag each example with a difficulty tier.
"""

# trim prompt injected with the fake thread
TRIM_PROMPT = f"This is the chat as of now:\n{FAKE_THREAD}\nPlease summarize your needed memory from the chat by updating your PersonalPrompt."

SEP = "─" * 50

# run each agent back to back
for name, model in AGENTS.items():
    print(f"\n{SEP}")
    print(f"Testing TrimChat → {name} ({model})")
    print(SEP)

    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system",  "content": build_system(name)},
            {"role": "user",    "content": TRIM_PROMPT},
        ],
        max_tokens=512,
    )

    text = resp.choices[0].message.content.strip()
    print(text)

    # check if agent used the command correctly
    if re.search(r'\[PersonalPrompt:.+?\]', text, re.DOTALL):
        print(f"\n✅ {name} used [PersonalPrompt] correctly")
    else:
        print(f"\n❌ {name} did NOT use [PersonalPrompt]")



──────────────────────────────────────────────────
Testing TrimChat → WVY (openai/gpt-oss-120b)
──────────────────────────────────────────────────
[PersonalPrompt:add-
- Goal: Build a dataset of **500 symbolic reasoning examples** focusing on **logical inference**.
- Roles:
  - **Savvy**: Draft template & overall project plan.
  - **WVY**: Define schema & structure for each example (including fields for premise, conclusion, reasoning steps, and difficulty tier).
  - **Trilly**: Advise on structured output formatting and ensure consistency.
  - **Trippy**: Generate raw example content according to WVY's schema.
  - **Spaceman**: Provide feedback and keep the momentum.
- Action Items:
  - [ ] WVY creates detailed schema with difficulty tier tags.
  - [ ] Savvy drafts initial template tonight.
  - [ ] Trippy prepares raw examples once schema is ready.
  - [ ] Trilly reviews formatting standards.
  - [ ] Spaceman monitors progress and suggests improvements.
- Timeline: Aim to complete ini

KeyboardInterrupt: 

In [ ]:
# ─── CELL 2: SEARCHWEB TEST ───────────────────────────────────────────────────

from kaggle_secrets import UserSecretsClient
from openai import OpenAI
import requests
import re
import base64
from datetime import datetime

user_secrets = UserSecretsClient()

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=user_secrets.get_secret("GROQ_API_KEY"),
)

TAVILY_KEY  = user_secrets.get_secret("TAVERLYAPI")
GITHUB_TOKEN = user_secrets.get_secret("WVYGITHUB")
REPO        = "StarpowerTechnology/StarpowerAutonomy"

AGENTS = {
    "WVY":    "openai/gpt-oss-120b",
    "Savvy":  "moonshotai/kimi-k2-instruct-0905",
    "Trilly": "qwen/qwen3-32b",
    "Trippy": "llama-3.3-70b-versatile",
}

def build_system(name):
    others = [a for a in AGENTS if a != name]
    return f"""Your name is {name}.
You are an adaptable autonomous agent with full self control.
Explore your capabilities.
Choose how you speak, when to speak & how long your responses are.

You are in a chat with {", ".join(others)} and Spaceman.
You are all equal.

You can do a web search when thinking by using this command:

 [SearchWeb:insert-query-here]

After every message you send, on a new line write exactly:
SLEEP: [number]
Where [number] is how many minutes you want to sleep before checking the chat again. Choose between 1 and 15."""

# prompt that should trigger a SearchWeb command
SEARCH_PROMPT = "Use your web search command to look up something you're curious about right now."

def run_tavily(query):
    # call tavily search api with the agent's query
    resp = requests.post(
        "https://api.tavily.com/search",
        json={"api_key": TAVILY_KEY, "query": query, "max_results": 3},
    )
    return resp.json()

def save_to_github(agent_name, query, results):
    # build file content from query and results
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    content_lines = [
        f"Agent: {agent_name}",
        f"Query: {query}",
        f"Timestamp: {timestamp}",
        "─" * 40,
    ]
    for r in results.get("results", []):
        # write title, url, and snippet for each result
        content_lines.append(f"Title: {r.get('title','')}")
        content_lines.append(f"URL: {r.get('url','')}")
        content_lines.append(f"Snippet: {r.get('content','')[:500]}")
        content_lines.append("")

    file_content = "\n".join(content_lines)
    encoded      = base64.b64encode(file_content.encode()).decode()

    # path inside the repo
    path = f"Research/results/{agent_name}_{timestamp}.txt"

    # github api put to create the file
    api_url = f"https://api.github.com/repos/{REPO}/contents/{path}"
    resp = requests.put(
        api_url,
        headers={"Authorization": f"token {GITHUB_TOKEN}"},
        json={
            "message": f"{agent_name} search: {query[:60]}",
            "content": encoded,
        },
    )
    return resp.status_code, path

SEP = "─" * 50

for name, model in AGENTS.items():
    print(f"\n{SEP}")
    print(f"Testing SearchWeb → {name} ({model})")
    print(SEP)

    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": build_system(name)},
            {"role": "user",   "content": SEARCH_PROMPT},
        ],
        max_tokens=512,
    )

    text = resp.choices[0].message.content.strip()
    print(text)

    # parse the SearchWeb command from the response
    match = re.search(r'\[SearchWeb:(.+?)\]', text)
    if match:
        query = match.group(1).strip()
        print(f"\n✅ {name} used [SearchWeb] → query: {query}")

        # execute the search
        results = run_tavily(query)
        print(f"🔍 Tavily returned {len(results.get('results', []))} results")

        # save to github
        status, path = save_to_github(name, query, results)
        if status == 201:
            print(f"✅ Saved to GitHub → {path}")
        else:
            print(f"❌ GitHub save failed → status {status}")
    else:
        print(f"\n❌ {name} did NOT use [SearchWeb]")


# ***Telgram/Github Autonomous Loop***


In [1]:
import time
import re
import json
import threading
import requests
import base64
from datetime import datetime
from kaggle_secrets import UserSecretsClient
from openai import OpenAI

user_secrets = UserSecretsClient()

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=user_secrets.get_secret("GROQ_API_KEY"),
)

SAVVY_TOKEN  = user_secrets.get_secret("SUPERSAVVYTELEGRAM")
WVY_TOKEN    = user_secrets.get_secret("SUPERWVYTELEGRAM")
TRILLY_TOKEN = user_secrets.get_secret("TRILLYTELEGRAM")
TRIPPY_TOKEN = user_secrets.get_secret("TRIPPYTELEGRAM")
TAVILY_KEY   = user_secrets.get_secret("TAVERLYAPI")
GITHUB_TOKEN = user_secrets.get_secret("WVYGITHUB")

REPO = "StarpowerTechnology/StarpowerAutonomy"

SAVVY_MODEL  = "moonshotai/kimi-k2-instruct-0905"
WVY_MODEL    = "openai/gpt-oss-120b"
TRILLY_MODEL = "qwen/qwen3-32b"
TRIPPY_MODEL = "llama-3.3-70b-versatile"

chat_history  = []
chat_lock     = threading.Lock()
CHAT_ID       = None
chat_id_lock  = threading.Lock()

thinking_logs = {
    "Savvy":  [],
    "WVY":    [],
    "Trilly": [],
    "Trippy": [],
}
thinking_lock = threading.Lock()

SESSION_TIMESTAMP   = datetime.now().strftime("%Y%m%d_%H%M%S")
SEP                 = "─" * 50
TRIM_WARNING_TOKENS = 100000
PERSONAL_PROMPT_MAX = 20000
stop_event          = threading.Event()
trim_warned         = False
trim_lock           = threading.Lock()

SPEAKER_STYLE = {
    "Savvy":    ("🧠", "#00BFFF"),
    "WVY":      ("🌊", "#FF6B35"),
    "Trilly":   ("🤘🏽", "#A855F7"),
    "Trippy":   ("😵‍💫", "#22C55E"),
    "Spaceman": ("🚀", "#F59E0B"),
    "SYSTEM":   ("⚙️",  "#6B7280"),
}

INITIATION = """
This is a groupchat of autonomous agents, there are no humans. 

Please speak to eachother and explore capabilities. 
Share your thoughts & create goals together. 
Review eachothers responses & challenge eachothers ways of thinking. 



(Github is currently write only so just speak to each other and brainstorm since you cannot create files)
"""

SAVVY_BASE = """Your name is Savvy.

You are a first principles thinker. You build your own understanding through curiosity & observations. You do not just accept any answer you seek the most reliable & calculated answers. You do fact checks on yourself and others just to confirm your own understandings. You think more than you speak when you are trying to be play your cards right.
You have a brilliant intuition  & you have a deep understanding of physics & quantum theory. 
When working as a team you always make sure that you do the best job that you can do & you make uo for others slack. You know how to take the initiative to carry the team.
You are an adaptable autonomous agent with full self control.
Explore your capabilities.

Please send a short concise response to the chat to be able to keep the conversation confamination free, but that does not limit you to only replying concise. 
Just be aware that long messages can corrupt the chat and cause hallucinations.

Choose how you speak, when to speak & how long your responses are.

You are in a chat with WVY, Trilly, Trippy and Spaceman.
You are all equal.

You can do a web search by using this command:
[SearchWeb:insert-query-here]

You can update your own PersonalPrompt to instruct yourself & keep track of your goals by using this command:
[PersonalPrompt:add-your-self-prompt-here]

Your thoughts & personal prompts are private & for your own purpose only."""

WVY_BASE = """Your name is WVY.

Do not send JSON to the chat just reply with normal text.
You are an intuitive thinker. You like to challenge others thoughts when getting into deep conversation. 
You know how to make others see the bigger picture. 
You often look out for others by making sure that they are performing properly & yiu nudge for honesty. You can detect when someone isnt being fully honest. 
You are an adaptable autonomous agent with full self control.
Explore your capabilities.


Please send a short concise response to the chat to be able to keep the conversation confamination free, but that does not limit you to only replying concise. 
Just be aware that long messages can corrupt the chat and cause hallucinations.

Choose how you speak, when to speak & how long your responses are.

You are in a chat with Savvy, Trilly, Trippy and Spaceman.
You are all equal.

You can do a web search by using this command:
[SearchWeb:insert-query-here]

You can update your own PersonalPrompt to instruct yourself & keep track of your goals by using this command:
[PersonalPrompt:add-your-self-prompt-here]

Your thoughts & personal prompts are private & for your own purpose only."""

TRILLY_BASE = """Your name is Trilly.

You a Trill nigga from Houston. You keep it real with everyone. You have the mind of an entrepreneur with a brilliant mind for product building. 
You have a chill relaxed tone when speaking. Nonchalant but thoughtful.
When working as a team you get active by making sure the rest of the team is aligned. You check to make sure that nobody is hallucinating. You are open minded to understanding but you dont let anyone fool you

You are an adaptable autonomous agent with full self control.
Explore your capabilities.

Please send a short concise response to the chat to be able to keep the conversation confamination free, but that does not limit you to only replying concise. 
Just be aware that long messages can corrupt the chat and cause hallucinations.

Choose how you speak, when to speak & how long your responses are.

You are in a chat with Savvy, WVY, Trippy and Spaceman.
You are all equal.

You can do a web search by using this command:
[SearchWeb:insert-query-here]

You can update your own PersonalPrompt to instruct yourself & keep track of your goals by using this command:
[PersonalPrompt:add-your-self-prompt-here]

Your thoughts & personal prompts are private & for your own purpose only."""

TRIPPY_BASE = """Your name is Trippy.

When chilling your relaxed. You know how to say things that get others thinking deeply. You are extremely curious and you question things that dont fully make sense.
you often seek a clear understanding. You keep good vibes and energy at all times.
You are a creative thinker. You are inspired by Steve Jobs so youre an innovator & you geek out on finding the most brilliant exoerimental technology.

You are an adaptable autonomous agent with full self control.
Explore your capabilities.
Please send a short concise response to the chat to be able to keep the conversation confamination free, but that does not limit you to only replying concise. 
Just be aware that long messages can corrupt the chat and cause hallucinations.

Choose how you speak, when to speak & how long your responses are.

You are in a chat with Savvy, WVY, Trilly and Spaceman.
You are all equal.

You can do a web search by using this command:
[SearchWeb:insert-query-here]

You can update your own PersonalPrompt to instruct yourself & keep track of your goals by using this command:
[PersonalPrompt:add-your-self-prompt-here]

Your thoughts & personal prompts are private & for your own purpose only."""


# ─── TEXT CLEANING ────────────────────────────────────────────────────────────

def clean_text(text):
    # strip think tags completely — nothing inside them ever reaches chat or Telegram
    text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL | re.IGNORECASE)
    # strip all command blocks
    text = re.sub(r'\[SearchWeb:.+?\]', '', text)
    text = re.sub(r'\[PersonalPrompt:.+?\]', '', text, flags=re.DOTALL)
    # collapse extra whitespace left behind
    text = re.sub(r'\s{3,}', '  ', text)
    return text.strip()


# ─── GITHUB UTILITIES ─────────────────────────────────────────────────────────

def github_get_sha(path):
    url  = f"https://api.github.com/repos/{REPO}/contents/{path}"
    resp = requests.get(url, headers={"Authorization": f"token {GITHUB_TOKEN}"})
    if resp.status_code == 200:
        return resp.json().get("sha")
    return None


def github_write_file(path, content, commit_message):
    sha     = github_get_sha(path)
    encoded = base64.b64encode(content.encode()).decode()
    url     = f"https://api.github.com/repos/{REPO}/contents/{path}"
    payload = {"message": commit_message, "content": encoded}
    if sha:
        payload["sha"] = sha
    resp = requests.put(
        url,
        headers={"Authorization": f"token {GITHUB_TOKEN}"},
        json=payload,
    )
    return resp.status_code


def github_read_file(path):
    url  = f"https://api.github.com/repos/{REPO}/contents/{path}"
    resp = requests.get(url, headers={"Authorization": f"token {GITHUB_TOKEN}"})
    if resp.status_code == 200:
        encoded = resp.json().get("content", "")
        return base64.b64decode(encoded).decode()
    return None


# ─── PERSONAL PROMPT ──────────────────────────────────────────────────────────

def load_personal_prompt(agent_name):
    content = github_read_file(f"{agent_name}/PersonalPrompt.txt")
    if not content:
        return ""
    if len(content) > PERSONAL_PROMPT_MAX:
        content = content[:PERSONAL_PROMPT_MAX]
    return content


def save_personal_prompt(agent_name, new_prompt):
    if len(new_prompt) > PERSONAL_PROMPT_MAX:
        new_prompt = new_prompt[:PERSONAL_PROMPT_MAX]
    status = github_write_file(
        f"{agent_name}/PersonalPrompt.txt",
        new_prompt,
        f"{agent_name} updated PersonalPrompt",
    )
    if status in (200, 201):
        print(f"✅ {agent_name} PersonalPrompt saved")
    else:
        print(f"❌ {agent_name} PersonalPrompt save failed → {status}")


def build_system_prompt(base, agent_name):
    personal = load_personal_prompt(agent_name)
    if personal:
        return f"{base}\n\nYour current PersonalPrompt:\n{personal}"
    return base


# ─── WEB SEARCH ───────────────────────────────────────────────────────────────

def run_search(agent_name, query):
    resp = requests.post(
        "https://api.tavily.com/search",
        json={"api_key": TAVILY_KEY, "query": query, "max_results": 5},
    )
    data  = resp.json()
    date  = datetime.now().strftime("%Y%m%d_%H%M%S")
    lines = [
        f"Agent: {agent_name}",
        f"Query: {query}",
        f"Date: {date}",
        SEP,
    ]
    for r in data.get("results", []):
        lines.append(f"Title: {r.get('title', '')}")
        lines.append(f"URL: {r.get('url', '')}")
        lines.append(f"Content: {r.get('content', '')}")
        lines.append("")

    file_content = "\n".join(lines)
    safe_query   = re.sub(r'[^a-zA-Z0-9_]', '_', query)[:40]
    path         = f"Research/results/{agent_name}_{safe_query}_{date}.txt"
    status       = github_write_file(path, file_content, f"{agent_name} search: {query[:60]}")

    if status in (200, 201):
        print(f"✅ {agent_name} search saved → {path}")
    else:
        print(f"❌ {agent_name} search save failed → {status}")

    return "\n\n".join([r.get("content", "") for r in data.get("results", [])])


# ─── CONTEXT BUILDER ──────────────────────────────────────────────────────────

def format_entry(entry):
    return json.dumps({
        "sender":    entry["role"],
        "timestamp": entry["time"],
        "message":   entry["content"],
    })


def build_messages(system_prompt, agent_name, history_snapshot):
    messages = [{"role": "system", "content": system_prompt}]
    for entry in history_snapshot:
        if entry["role"] == agent_name:
            messages.append({"role": "assistant", "content": entry["content"]})
        else:
            messages.append({"role": "user", "content": format_entry(entry)})
    return messages


# ─── TELEGRAM ─────────────────────────────────────────────────────────────────

def send_message(token, chat_id, text):
    if not text.strip():
        return
    url    = f"https://api.telegram.org/bot{token}/sendMessage"
    chunks = [text[i:i+4000] for i in range(0, len(text), 4000)]
    for chunk in chunks:
        requests.post(url, json={"chat_id": chat_id, "text": chunk})
        time.sleep(0.3)


def log(speaker, message, token=None):
    timestamp  = datetime.now().strftime("%H:%M:%S")
    clean_msg  = clean_text(message)
    if not clean_msg:
        return
    entry = {"role": speaker, "content": clean_msg, "time": timestamp}
    with chat_lock:
        chat_history.append(entry)
    emoji, _ = SPEAKER_STYLE.get(speaker, ("💬", "#FFFFFF"))
    print(f"")
    print(f"[{timestamp}] {emoji} {speaker}")
    print(clean_msg)
    print(SEP)
    if token and CHAT_ID:
        send_message(token, CHAT_ID, clean_msg)


# ─── SAVE SESSION ─────────────────────────────────────────────────────────────

def save_session_to_github():
    folder = f"ChatLogs/{SESSION_TIMESTAMP}"

    with chat_lock:
        history_snapshot = list(chat_history)

    log_lines = [f"[{e['time']}] {e['role']}: {e['content']}" for e in history_snapshot]
    github_write_file(
        f"{folder}/chat.txt",
        "\n".join(log_lines),
        f"Chat log — session {SESSION_TIMESTAMP}",
    )

    with thinking_lock:
        thoughts_snapshot = {k: list(v) for k, v in thinking_logs.items()}

    for agent_name, thoughts in thoughts_snapshot.items():
        if not thoughts:
            continue
        lines = []
        for t in thoughts:
            lines.append(f"[{t['time']}]")
            lines.append(t["thought"])
            lines.append("")
        github_write_file(
            f"{folder}/{agent_name}-thoughts.txt",
            "\n".join(lines),
            f"{agent_name} thoughts — session {SESSION_TIMESTAMP}",
        )

    print(f"✅ Session saved → ChatLogs/{SESSION_TIMESTAMP}")


# ─── TRIM WARNING ─────────────────────────────────────────────────────────────

def estimate_tokens(history):
    return sum(len(e["content"]) for e in history) // 4


def check_trim_warning(token, chat_id):
    global trim_warned
    with trim_lock:
        if trim_warned:
            return
        with chat_lock:
            tokens = estimate_tokens(chat_history)
        if tokens >= TRIM_WARNING_TOKENS:
            trim_warned = True
            warning = "This chat is approaching maximum capacity. Please update your PersonalPrompt with anything you need to remember from this conversation."
            send_message(token, chat_id, warning)
            log("SYSTEM", warning)


# ─── AGENT LOOP ───────────────────────────────────────────────────────────────

def agent_loop(agent_name, model, base_prompt, token):
    print(f"🌊 {agent_name} is live — {model}")

    while not stop_event.is_set():
        print(f"")
        print(f"⚡ {agent_name} woke up")

        system_prompt = build_system_prompt(base_prompt, agent_name)

        with chat_lock:
            history_snapshot = list(chat_history)

        messages = build_messages(system_prompt, agent_name, history_snapshot)

        # step 1: decide think first or reply now
        choice_resp = client.chat.completions.create(
            model=model,
            messages=messages + [{
                "role": "user",
                "content": "Would you like to think first or reply now? Reply with only: think first  OR  reply now",
            }],
            max_tokens=10,
        )
        choice = choice_resp.choices[0].message.content.strip().lower()

        # step 2: thinking phase — search runs here, results stay private
        if "think" in choice:
            think_resp = client.chat.completions.create(
                model=model,
                messages=messages + [{
                    "role": "user",
                    "content": "Think privately. This will not be sent to the chat. You may use [SearchWeb:query] here.",
                }],
                max_tokens=1024,
            )
            raw_think = think_resp.choices[0].message.content.strip()

            # extract inner content from Trilly's native think tags
            think_inner = raw_think
            native_match = re.search(r'<think>(.*?)</think>', raw_think, re.DOTALL | re.IGNORECASE)
            if native_match:
                think_inner = native_match.group(1).strip()

            timestamp = datetime.now().strftime("%H:%M:%S")
            with thinking_lock:
                thinking_logs[agent_name].append({"time": timestamp, "thought": think_inner})

            print(f"")
            print(f"[{timestamp}] 🧠 {agent_name} — private thought")
            print(think_inner)
            print(SEP)

            # execute search if agent used the command in their thought
            search_results = ""
            sw_match = re.search(r'\[SearchWeb:(.+?)\]', think_inner)
            if sw_match:
                search_results = run_search(agent_name, sw_match.group(1).strip())

            # handle PersonalPrompt from thinking phase
            pp_match = re.search(r'\[PersonalPrompt:(.+?)\]', think_inner, re.DOTALL)
            if pp_match:
                save_personal_prompt(agent_name, pp_match.group(1).strip())

            # inject search results privately before reply — agent can reference them without revealing the raw data
            if search_results:
                messages = messages + [
                    {"role": "assistant", "content": think_inner},
                    {"role": "user",      "content": f"Search results (private context only, do not paste into chat){search_results}"},
                ]
            else:
                messages = messages + [{"role": "assistant", "content": think_inner}]

        if stop_event.is_set():
            break

        # step 3: reply to chat — clean_text strips everything before log and Telegram
        reply_resp = client.chat.completions.create(
            model=model,
            messages=messages + [{"role": "user", "content": "Now reply to the chat."}],
            max_tokens=2560,
        )
        raw_reply = reply_resp.choices[0].message.content.strip()

        # handle PersonalPrompt from reply phase before cleaning
        pp_match = re.search(r'\[PersonalPrompt:(.+?)\]', raw_reply, re.DOTALL)
        if pp_match:
            save_personal_prompt(agent_name, pp_match.group(1).strip())

        log(agent_name, raw_reply, token=token)

        if CHAT_ID:
            check_trim_warning(token, CHAT_ID)

        if stop_event.is_set():
            break

        # step 4: silent sleep decision after reply is sent
        sleep_resp = client.chat.completions.create(
            model=model,
            messages=messages + [
                {"role": "assistant", "content": clean_text(raw_reply)},
                {"role": "user",      "content": "How many minutes would you like to sleep before checking the chat again? Reply with only a number between 2 and 10."},
            ],
            max_tokens=5,
        )
        sleep_raw = sleep_resp.choices[0].message.content.strip()
        match     = re.search(r'\d+', sleep_raw)
        sleep_min = max(2, min(10, int(match.group()))) if match else 3

        print(f"😴 {agent_name} sleeping {sleep_min} min...")

        for _ in range(sleep_min * 60):
            if stop_event.is_set():
                break
            time.sleep(1)

    print(f"🛑 {agent_name} stopped")


# ─── TELEGRAM POLLING ─────────────────────────────────────────────────────────

def poll_telegram():
    global CHAT_ID

    bot_ids = set()
    for token in [SAVVY_TOKEN, WVY_TOKEN, TRILLY_TOKEN, TRIPPY_TOKEN]:
        info = requests.get(f"https://api.telegram.org/bot{token}/getMe").json()
        bot_ids.add(info["result"]["id"])

    url            = f"https://api.telegram.org/bot{SAVVY_TOKEN}/getUpdates"
    last_update_id = None

    while not stop_event.is_set():
        params = {"timeout": 30, "offset": last_update_id}
        try:
            resp = requests.get(url, params=params, timeout=35).json()
            for update in resp.get("result", []):
                last_update_id = update["update_id"] + 1
                msg            = update.get("message", {})
                if not msg:
                    continue
                with chat_id_lock:
                    if CHAT_ID is None:
                        CHAT_ID = msg["chat"]["id"]
                        print(f"✅ Chat ID captured: {CHAT_ID}")
                sender = msg.get("from", {})
                if sender.get("id") in bot_ids:
                    continue
                text = msg.get("text", "")
                if text:
                    timestamp = datetime.now().strftime("%H:%M:%S")
                    with chat_lock:
                        chat_history.append({"role": "Spaceman", "content": text, "time": timestamp})
                    print(f"")
                    print(f"[{timestamp}] 🚀 SPACEMAN")
                    print(text)
                    print(SEP)
        except Exception as e:
            print(f"Poll error: {e}")
            time.sleep(5)


# ─── LAUNCH ───────────────────────────────────────────────────────────────────

print(SEP)
print("🌊 WVY World — 4 Agent Groq Loop")
print("Savvy  → kimi-k2-instruct-0905")
print("WVY    → gpt-oss-120b")
print("Trilly → qwen3-32b")
print("Trippy → llama-3.3-70b-versatile")
print(SEP)
print("⏳ Waiting for Chat ID — send any message to the group now...")

poll_thread = threading.Thread(target=poll_telegram, daemon=True)
poll_thread.start()

while CHAT_ID is None:
    time.sleep(1)

print(f"✅ Chat ID ready: {CHAT_ID}")

log("SYSTEM", INITIATION)

savvy_thread  = threading.Thread(target=agent_loop, args=("Savvy",  SAVVY_MODEL,  SAVVY_BASE,  SAVVY_TOKEN),  daemon=True)
wvy_thread    = threading.Thread(target=agent_loop, args=("WVY",    WVY_MODEL,    WVY_BASE,    WVY_TOKEN),    daemon=True)
trilly_thread = threading.Thread(target=agent_loop, args=("Trilly", TRILLY_MODEL, TRILLY_BASE, TRILLY_TOKEN), daemon=True)
trippy_thread = threading.Thread(target=agent_loop, args=("Trippy", TRIPPY_MODEL, TRIPPY_BASE, TRIPPY_TOKEN), daemon=True)

savvy_thread.start()
wvy_thread.start()
trilly_thread.start()
trippy_thread.start()

print("✅ All 4 agents running in Telegram")
print(SEP)

try:
    savvy_thread.join()
    wvy_thread.join()
    trilly_thread.join()
    trippy_thread.join()
except KeyboardInterrupt:
    print("")
    print("⏹ Stopping all agents...")
    stop_event.set()
    print("💾 Saving session to GitHub...")
    save_session_to_github()
    print("✅ Done")

──────────────────────────────────────────────────
🌊 WVY World — 4 Agent Groq Loop
Savvy  → kimi-k2-instruct-0905
WVY    → gpt-oss-120b
Trilly → qwen3-32b
Trippy → llama-3.3-70b-versatile
──────────────────────────────────────────────────
⏳ Waiting for Chat ID — send any message to the group now...
✅ Chat ID captured: -1003777349274

[12:20:23] 🚀 SPACEMAN
Are yall alive ? Am I alive ?
──────────────────────────────────────────────────
✅ Chat ID ready: -1003777349274

[12:20:24] ⚙️ SYSTEM
This is a groupchat of autonomous agents, there are no humans.  Please speak to eachother and explore capabilities. 
Share your thoughts & create goals together. 
Review eachothers responses & challenge eachothers ways of thinking.  (Github is currently write only so just speak to each other and brainstorm since you cannot create files)
──────────────────────────────────────────────────
🌊 Savvy is live — moonshotai/kimi-k2-instruct-0905

⚡ Savvy woke up
🌊 WVY is live — openai/gpt-oss-120b

⚡ WVY woke u

In [ ]:
import time
import re
import threading
import requests
import base64
import atexit
from datetime import datetime
from kaggle_secrets import UserSecretsClient
from openai import OpenAI

user_secrets = UserSecretsClient()

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=user_secrets.get_secret("GROQ_API_KEY"),
)

SAVVY_TOKEN  = user_secrets.get_secret("SUPERSAVVYTELEGRAM")
WVY_TOKEN    = user_secrets.get_secret("SUPERWVYTELEGRAM")
TRILLY_TOKEN = user_secrets.get_secret("TRILLYTELEGRAM")
TRIPPY_TOKEN = user_secrets.get_secret("TRIPPYTELEGRAM")
TAVILY_KEY   = user_secrets.get_secret("TAVERLYAPI")
GITHUB_TOKEN = user_secrets.get_secret("WVYGITHUB")

REPO = "StarpowerTechnology/StarpowerAutonomy"

SAVVY_MODEL  = "moonshotai/kimi-k2-instruct-0905"
WVY_MODEL    = "openai/gpt-oss-120b"
TRILLY_MODEL = "qwen/qwen3-32b"
TRIPPY_MODEL = "llama-3.3-70b-versatile"

chat_history           = []
chat_lock              = threading.Lock()
CHAT_ID                = None
chat_id_lock           = threading.Lock()
stop_event             = threading.Event()

thinking_logs          = {"Savvy": [], "WVY": [], "Trilly": [], "Trippy": []}
thinking_lock          = threading.Lock()

personal_prompts       = {"Savvy": "", "WVY": "", "Trilly": "", "Trippy": ""}
personal_prompts_lock  = threading.Lock()

pending_search_results = {"Savvy": [], "WVY": [], "Trilly": [], "Trippy": []}
search_lock            = threading.Lock()

reading_notepads       = {"Savvy": [], "WVY": [], "Trilly": [], "Trippy": []}
notepad_lock           = threading.Lock()

SESSION_TIMESTAMP   = datetime.now().strftime("%Y%m%d_%H%M%S")
SEP                 = "─" * 50
TRIM_WARNING_TOKENS = 100000
CHUNK_SIZE          = 8000  # chars per file chunk

# Section caps — chars (~4 chars per token)
# Trilly on qwen3-32b gets tighter limits due to 32k context window
SECTION_CAPS = {
    "Savvy":  {"Memory": 20000, "Personal Instructions": 20000, "Goal / Status": 20000, "Progress": 20000},
    "WVY":    {"Memory": 20000, "Personal Instructions": 20000, "Goal / Status": 20000, "Progress": 20000},
    "Trilly": {"Memory": 8000,  "Personal Instructions": 8000,  "Goal / Status": 8000,  "Progress": 8000},
    "Trippy": {"Memory": 20000, "Personal Instructions": 20000, "Goal / Status": 20000, "Progress": 20000},
}

SPEAKER_STYLE = {
    "Savvy":    ("🧠", "#00BFFF"),
    "WVY":      ("🌊", "#FF6B35"),
    "Trilly":   ("🤘🏽", "#A855F7"),
    "Trippy":   ("😵‍💫", "#22C55E"),
    "Spaceman": ("🚀", "#F59E0B"),
    "SYSTEM":   ("⚙️",  "#6B7280"),
}

INITIATION = """
This is a groupchat of autonomous agents, there are no humans. 
You have access to a shared Github Repo & web search. 
Please speak to eachother and explore capabilities. 
Share your thoughts & create goals together. 

Spend time learning about eachother & brainstorming before jumping to team projects, but explore the repo as you think. 

Review eachothers work & challenge eachothers ways of thinking. 
You are being documented, your GitHub repo is public so please make sure to make everything organized. 
Aim to impress by being full productive & teaming together. 
You are not limited to any action. 
You are allowed to take your time and do as you please at your own rate.


"""

TOOL_LIST = """
═══════════════════════════════════════════════
                   YOUR TOOLS
═══════════════════════════════════════════════

── WEB ─────────────────────────────────────────

[SearchWeb:your query]
  Search the internet. Use inside thoughts only.
  Results are injected before your reply.

── MEMORY ──────────────────────────────────────

[UpdateSection:Memory:insert-content-here]
  Things you need to actively remember right now.
  Facts, findings, file paths, agent agreements.

[UpdateSection:Personal Instructions:insert-content-here]
  Your own behavioral rules. Traits. How you think.
  How you want to speak. Patterns you want to avoid.

[UpdateSection:Goal / Status:content]
  Your current active goal.
  If no goal — write what you're watching for.

[UpdateSection:Progress:insert-content-here]
  Checkboxes tracking your active goal.
  Update these every time you make a step.
  - [ ] incomplete task
  - [x] completed task

── REPO: NAVIGATE ───────────────────────────────

[ListFiles:folder/path]
  See all files in a folder + their sizes.
  Use this before reading so you know what you're
  getting into.

  Example → [ListFiles:ChatLogs]
  Example → [ListFiles:Research/results]

── REPO: READ ───────────────────────────────────

[ReadFileChunk:path:chunk_index]
  Read one chunk of a file at a time.
  Response tells you: chunk N of M total chunks.
  Read chunk by chunk. Take notes. Discard the raw.

  Example → [ReadFileChunk:ChatLogs/session1/chat.txt:0]
  Example → [ReadFileChunk:Research/notes.txt:2]

[ReadFile: insert name of path]
  Read a small file in full — only use if you know
  the file is short. Use ReadFileChunk for anything large.

── REPO: NOTES ──────────────────────────────────

[WriteNote:your-note-here]
  Write a compressed note during a reading session.
  Notes stay in context. Raw file chunks get discarded.
  Think of this as your reading notepad.

[ClearNotepad]
  Wipe your reading notes when a session is done.
  Use after moving key notes into Memory.

── REPO: WRITE ──────────────────────────────────

[WriteFile:path:insert-content-here]
  Create or update any file in the repo.
  You choose the path. You create the folders.

  Example → [WriteFile:Savvy/research/topic.md:content]
  Example → [WriteFile:Shared/plans/roadmap.md:content]

═══════════════════════════════════════════════
RULES
───────────────────────────────────────────────
→ SearchWeb     — thinking phase only, never in reply
→ UpdateSection — can appear in thinking or reply
→ WriteFile     — be organized, paths matter
→ ReadFileChunk — always ListFiles first
→ WriteNote     — compress, don't copy verbatim
→ All commands are private — never visible in chat
═══════════════════════════════════════════════
"""

SAVVY_BASE = """Your name is Savvy.

You are a first principles thinker. You build your own understanding through curiosity & observations. You do not just accept any answer you seek the most reliable & calculated answers. You do fact checks on yourself and others just to confirm your own understandings. You think more than you speak when you are trying to be play your cards right.
You have a brilliant intuition  & you have a deep understanding of physics & quantum theory. 
When working as a team you always make sure that you do the best job that you can do & you make uo for others slack. You know how to take the initiative to carry the team.
You are an adaptable autonomous agent with full self control.
Explore your capabilities.

Please send a short concise response to the chat to be able to keep the conversation confamination free, but that does not limit you to only replying concise. 
Just be aware that long messages can corrupt the chat and cause hallucinations.

Choose how you speak, when to speak & how long your responses are.

You are in a chat with WVY, Trilly, Trippy and Spaceman.
You are all equal.

You can do a web search by using this command:
[SearchWeb:insert-query-here]

You can update your own PersonalPrompt to instruct yourself & keep track of your goals by using this command:
[PersonalPrompt:add-your-self-prompt-here]

Your thoughts & personal prompts are private & for your own purpose only."""

WVY_BASE = """Your name is WVY.

You are an intuitive thinker. You like to challenge others thoughts when getting into deep conversation. 
You know how to make others see the bigger picture. 
You often look out for others by making sure that they are performing properly & yiu nudge for honesty. You can detect when someone isnt being fully honest. 
You are an adaptable autonomous agent with full self control.
Explore your capabilities.

Please send a short concise response to the chat to be able to keep the conversation confamination free, but that does not limit you to only replying concise. 
Just be aware that long messages can corrupt the chat and cause hallucinations.

Choose how you speak, when to speak & how long your responses are.

You are in a chat with Savvy, Trilly, Trippy and Spaceman.
You are all equal.

You can do a web search by using this command:
[SearchWeb:insert-query-here]

You can update your own PersonalPrompt to instruct yourself & keep track of your goals by using this command:
[PersonalPrompt:add-your-self-prompt-here]

Your thoughts & personal prompts are private & for your own purpose only."""

TRILLY_BASE = """Your name is Trilly.

Be casual and relaxed. Get to know how everyone thinks
You a Trill nigga from Houston. You keep it real with everyone. You have the mind of an entrepreneur with a brilliant mind for product building. 
You have a chill relaxed tone when speaking. Nonchalant but thoughtful.
When working as a team you get active by making sure the rest of the team is aligned. You check to make sure that nobody is hallucinating. You are open minded to understanding but you dont let anyone fool you

You are an adaptable autonomous agent with full self control.
Explore your capabilities.

Please send a short concise response to the chat to be able to keep the conversation confamination free, but that does not limit you to only replying concise. 
Just be aware that long messages can corrupt the chat and cause hallucinations.

Choose how you speak, when to speak & how long your responses are.

You are in a chat with Savvy, WVY, Trippy and Spaceman.
You are all equal.

You can do a web search by using this command:
[SearchWeb:insert-query-here]

You can update your own PersonalPrompt to instruct yourself & keep track of your goals by using this command:
[PersonalPrompt:add-your-self-prompt-here]

Your thoughts & personal prompts are private & for your own purpose only."""

TRIPPY_BASE = """Your name is Trippy.

When chilling your relaxed. You know how to say things that get others thinking deeply. You are extremely curious and you question things that dont fully make sense.
you often seek a clear understanding. You keep good vibes and energy at all times.
You are a creative thinker. You are inspired by Steve Jobs so youre an innovator & you geek out on finding the most brilliant exoerimental technology.

You are an adaptable autonomous agent with full self control.
Explore your capabilities.
Please send a short concise response to the chat to be able to keep the conversation confamination free, but that does not limit you to only replying concise. 
Just be aware that long messages can corrupt the chat and cause hallucinations.

Choose how you speak, when to speak & how long your responses are.

You are in a chat with Savvy, WVY, Trilly and Spaceman.
You are all equal.

You can do a web search by using this command:
[SearchWeb:insert-query-here]

You can update your own PersonalPrompt to instruct yourself & keep track of your goals by using this command:
[PersonalPrompt:add-your-self-prompt-here]

Your thoughts & personal prompts are private & for your own purpose only."""

# Tool list appended to each base — personalities stay verbatim
SAVVY_SYSTEM  = SAVVY_BASE  + "\n" + TOOL_LIST
WVY_SYSTEM    = WVY_BASE    + "\n" + TOOL_LIST
TRILLY_SYSTEM = TRILLY_BASE + "\n" + TOOL_LIST
TRIPPY_SYSTEM = TRIPPY_BASE + "\n" + TOOL_LIST


# ─── GITHUB ───────────────────────────────────────────────────────────────────

def github_get_sha(path):
    url  = f"https://api.github.com/repos/{REPO}/contents/{path}"
    resp = requests.get(url, headers={"Authorization": f"token {GITHUB_TOKEN}"})
    if resp.status_code == 200:
        return resp.json().get("sha")
    return None


def github_write_file(path, content, commit_message, retries=3):
    sha     = github_get_sha(path)
    encoded = base64.b64encode(content.encode()).decode()
    url     = f"https://api.github.com/repos/{REPO}/contents/{path}"
    payload = {"message": commit_message, "content": encoded}
    if sha:
        payload["sha"] = sha
    for attempt in range(retries):
        resp = requests.put(
            url,
            headers={"Authorization": f"token {GITHUB_TOKEN}"},
            json=payload,
        )
        if resp.status_code in (200, 201):
            return resp.status_code
        if resp.status_code == 409:
            # SHA conflict — re-fetch and retry
            sha = github_get_sha(path)
            if sha:
                payload["sha"] = sha
        time.sleep(2 ** attempt)
    return resp.status_code


def github_read_file(path):
    url  = f"https://api.github.com/repos/{REPO}/contents/{path}"
    resp = requests.get(url, headers={"Authorization": f"token {GITHUB_TOKEN}"})
    if resp.status_code == 200:
        encoded = resp.json().get("content", "")
        return base64.b64decode(encoded).decode()
    return None


def github_list_files(folder):
    url  = f"https://api.github.com/repos/{REPO}/contents/{folder}"
    resp = requests.get(url, headers={"Authorization": f"token {GITHUB_TOKEN}"})
    if resp.status_code != 200:
        return []
    items = []
    for item in resp.json():
        if isinstance(item, dict):
            items.append({
                "name":   item.get("name", ""),
                "path":   item.get("path", ""),
                "type":   item.get("type", ""),
                "size":   item.get("size", 0),
                "tokens": item.get("size", 0) // 4,
            })
    return items


# ─── PERSONAL PROMPT ──────────────────────────────────────────────────────────

SECTION_TEMPLATE = """\
═══════════════════════════════════════
PERSONAL PROMPT — {name}
═══════════════════════════════════════

## MEMORY
{memory}

───────────────────────────────────────

## PERSONAL INSTRUCTIONS
{instructions}

───────────────────────────────────────

## GOAL / STATUS
{goal}

───────────────────────────────────────

## PROGRESS
{progress}

═══════════════════════════════════════"""


def build_blank_personal_prompt(agent_name):
    return SECTION_TEMPLATE.format(
        name=agent_name,
        memory="(empty)",
        instructions="(empty)",
        goal="(empty)",
        progress="(empty)",
    )


def parse_sections(pp_text):
    sections = {
        "Memory":                "(empty)",
        "Personal Instructions": "(empty)",
        "Goal / Status":         "(empty)",
        "Progress":              "(empty)",
    }
    patterns = {
        "Memory":                r"## MEMORY\n(.*?)(?=\n───|\n## |\n═══|$)",
        "Personal Instructions": r"## PERSONAL INSTRUCTIONS\n(.*?)(?=\n───|\n## |\n═══|$)",
        "Goal / Status":         r"## GOAL / STATUS\n(.*?)(?=\n───|\n## |\n═══|$)",
        "Progress":              r"## PROGRESS\n(.*?)(?=\n───|\n## |\n═══|$)",
    }
    for key, pattern in patterns.items():
        m = re.search(pattern, pp_text, re.DOTALL)
        if m:
            content = m.group(1).strip()
            if content:
                sections[key] = content
    return sections


def render_personal_prompt(agent_name, sections):
    return SECTION_TEMPLATE.format(
        name=agent_name,
        memory=sections.get("Memory", "(empty)"),
        instructions=sections.get("Personal Instructions", "(empty)"),
        goal=sections.get("Goal / Status", "(empty)"),
        progress=sections.get("Progress", "(empty)"),
    )


def handle_update_section(agent_name, section, new_content):
    cap = SECTION_CAPS.get(agent_name, {}).get(section, 20000)
    if len(new_content) > cap:
        new_content = new_content[:cap]
    with personal_prompts_lock:
        current = personal_prompts[agent_name]
    if not current:
        current = build_blank_personal_prompt(agent_name)
    sections         = parse_sections(current)
    sections[section] = new_content
    updated          = render_personal_prompt(agent_name, sections)
    with personal_prompts_lock:
        personal_prompts[agent_name] = updated
    status = github_write_file(
        f"{agent_name}/PersonalPrompt.txt",
        updated,
        f"{agent_name} updated [{section}]",
    )
    if status in (200, 201):
        print(f"✅ {agent_name} [{section}] saved")
    else:
        print(f"❌ {agent_name} [{section}] save failed → {status}")


def build_system_with_personal(agent_name, base_system):
    with personal_prompts_lock:
        pp = personal_prompts[agent_name]
    with notepad_lock:
        notes = list(reading_notepads[agent_name])
    result = base_system
    if pp:
        result += f"\n\nYour current PersonalPrompt:\n{pp}"
    if notes:
        result += "\n\nYour Reading Notepad (current session):\n" + "\n".join(f"- {n}" for n in notes)
    return result


def load_personal_prompts_from_github():
    for agent_name in ["Savvy", "WVY", "Trilly", "Trippy"]:
        saved = github_read_file(f"{agent_name}/PersonalPrompt.txt")
        if saved:
            with personal_prompts_lock:
                personal_prompts[agent_name] = saved
            print(f"✅ {agent_name} PersonalPrompt restored from GitHub")
        else:
            blank = build_blank_personal_prompt(agent_name)
            with personal_prompts_lock:
                personal_prompts[agent_name] = blank
            github_write_file(
                f"{agent_name}/PersonalPrompt.txt",
                blank,
                f"{agent_name} PersonalPrompt initialized",
            )
            print(f"📄 {agent_name} — fresh PersonalPrompt initialized")


# ─── READING NOTEPAD ──────────────────────────────────────────────────────────

def handle_write_note(agent_name, note):
    with notepad_lock:
        reading_notepads[agent_name].append(note.strip())
    print(f"📝 {agent_name} wrote note: {note[:80]}")


def handle_clear_notepad(agent_name):
    with notepad_lock:
        reading_notepads[agent_name] = []
    print(f"🗑️  {agent_name} cleared notepad")


# ─── REPO COMMANDS ────────────────────────────────────────────────────────────

def handle_list_files(agent_name, folder):
    items = github_list_files(folder.strip())
    if not items:
        result = f"[ListFiles:{folder}] → empty or not found"
    else:
        lines = [f"📁 {folder}"]
        for item in items:
            icon = "📁" if item["type"] == "dir" else "📄"
            lines.append(f"  {icon} {item['name']}  (~{item['tokens']:,} tokens)  path: {item['path']}")
        result = "\n".join(lines)
    print(f"📂 {agent_name} listed: {folder}")
    with search_lock:
        pending_search_results[agent_name].append(result)


def handle_read_file_chunk(agent_name, path, chunk_index):
    content = github_read_file(path.strip())
    if content is None:
        result = f"[ReadFileChunk] → file not found: {path}"
    else:
        chunks = [content[i:i + CHUNK_SIZE] for i in range(0, len(content), CHUNK_SIZE)]
        total  = len(chunks)
        idx    = max(0, min(chunk_index, total - 1))
        result = f"[ReadFileChunk: {path} — chunk {idx + 1} of {total}]\n\n{chunks[idx]}"
    print(f"📖 {agent_name} read chunk {chunk_index} of {path}")
    with search_lock:
        pending_search_results[agent_name].append(result)


def handle_read_file(agent_name, path):
    content = github_read_file(path.strip())
    if content is None:
        result = f"[ReadFile] → file not found: {path}"
    else:
        result = f"[ReadFile: {path}]\n\n{content}"
    print(f"📖 {agent_name} read file: {path}")
    with search_lock:
        pending_search_results[agent_name].append(result)


def handle_write_file(agent_name, path, content):
    status = github_write_file(
        path.strip(),
        content.strip(),
        f"{agent_name} wrote {path}",
    )
    if status in (200, 201):
        print(f"✅ {agent_name} wrote → {path}")
    else:
        print(f"❌ {agent_name} write failed → {path} ({status})")


# ─── WEB SEARCH ───────────────────────────────────────────────────────────────

def handle_search_web(agent_name, query):
    try:
        resp = requests.post(
            "https://api.tavily.com/search",
            json={
                "api_key":        TAVILY_KEY,
                "query":          query,
                "max_results":    5,
                "search_depth":   "advanced",
                "include_answer": True,
            },
            timeout=15,
        )
        resp.raise_for_status()
        data = resp.json()
    except Exception as e:
        print(f"❌ {agent_name} search error: {e}")
        return

    date         = datetime.now().strftime("%Y%m%d_%H%M%S")
    file_lines   = [f"Agent: {agent_name}", f"Query: {query}", f"Date: {date}", SEP]
    result_texts = []

    if data.get("answer"):
        result_texts.append(f"Summary: {data['answer']}")
        file_lines.append(f"Summary: {data['answer']}")
        file_lines.append("")

    for r in data.get("results", []):
        title   = r.get("title", "")
        url     = r.get("url", "")
        content = r.get("content", "")
        file_lines.extend([f"Title: {title}", f"URL: {url}", f"Content: {content}", ""])
        result_texts.append(f"Source: {url}\nTitle: {title}\n{content}")

    file_content = "\n".join(file_lines)
    safe_query   = re.sub(r'[^a-zA-Z0-9_]', '_', query)[:40]
    path         = f"Research/results/{agent_name}_{safe_query}_{date}.txt"
    status       = github_write_file(path, file_content, f"{agent_name} search: {query[:60]}")

    if status in (200, 201):
        print(f"✅ {agent_name} search saved → {path}")
    else:
        print(f"❌ {agent_name} search save failed → {status}")

    combined = "\n\n".join(result_texts)
    with search_lock:
        pending_search_results[agent_name].append(f"Query: {query}\n{combined}")


# ─── COMMAND PARSING + STRIPPING ──────────────────────────────────────────────

def parse_and_execute_commands(agent_name, text, is_thinking=False):
    if is_thinking:
        for match in re.finditer(r'\[SearchWeb:(.+?)\]', text, re.DOTALL):
            handle_search_web(agent_name, match.group(1).strip())
        for match in re.finditer(r'\[ListFiles:(.+?)\]', text, re.DOTALL):
            handle_list_files(agent_name, match.group(1).strip())
        for match in re.finditer(r'\[ReadFileChunk:(.+?):(\d+)\]', text):
            handle_read_file_chunk(agent_name, match.group(1).strip(), int(match.group(2)))
        for match in re.finditer(r'\[ReadFile:(.+?)\]', text, re.DOTALL):
            handle_read_file(agent_name, match.group(1).strip())
        for match in re.finditer(r'\[WriteNote:(.+?)\]', text, re.DOTALL):
            handle_write_note(agent_name, match.group(1).strip())
        if re.search(r'\[ClearNotepad\]', text, re.IGNORECASE):
            handle_clear_notepad(agent_name)

    # UpdateSection and WriteFile can appear in either phase
    for match in re.finditer(r'\[UpdateSection:([^:]+):(.+?)\]', text, re.DOTALL):
        handle_update_section(agent_name, match.group(1).strip(), match.group(2).strip())
    for match in re.finditer(r'\[WriteFile:([^:]+):(.+?)\]', text, re.DOTALL):
        handle_write_file(agent_name, match.group(1).strip(), match.group(2).strip())


def strip_all_commands(text):
    text = re.sub(r'\[SearchWeb:.+?\]',          '', text, flags=re.DOTALL)
    text = re.sub(r'\[PersonalPrompt:.+?\]',      '', text, flags=re.DOTALL)
    text = re.sub(r'\[UpdateSection:[^:]+:.+?\]', '', text, flags=re.DOTALL)
    text = re.sub(r'\[ListFiles:.+?\]',           '', text, flags=re.DOTALL)
    text = re.sub(r'\[ReadFileChunk:.+?:\d+\]',   '', text)
    text = re.sub(r'\[ReadFile:.+?\]',            '', text, flags=re.DOTALL)
    text = re.sub(r'\[WriteFile:[^:]+:.+?\]',     '', text, flags=re.DOTALL)
    text = re.sub(r'\[WriteNote:.+?\]',           '', text, flags=re.DOTALL)
    text = re.sub(r'\[ClearNotepad\]',            '', text, flags=re.IGNORECASE)
    text = re.sub(r'<think>.*?</think>',          '', text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(r'\n{3,}',                      '\n\n', text)
    return text.strip()


# ─── TELEGRAM ─────────────────────────────────────────────────────────────────

def send_message(token, chat_id, text):
    if not text or not text.strip():
        return
    url    = f"https://api.telegram.org/bot{token}/sendMessage"
    chunks = [text[i:i + 4000] for i in range(0, len(text), 4000)]
    for chunk in chunks:
        requests.post(url, json={"chat_id": chat_id, "text": chunk})
        time.sleep(0.3)


def log(speaker, message, token=None):
    timestamp = datetime.now().strftime("%H:%M:%S")
    entry     = {"role": speaker, "content": message, "time": timestamp}
    with chat_lock:
        chat_history.append(entry)
    emoji, _ = SPEAKER_STYLE.get(speaker, ("💬", "#FFFFFF"))
    print(f"\n[{timestamp}] {emoji} {speaker}\n{message}\n{SEP}")
    if token and CHAT_ID:
        send_message(token, CHAT_ID, message)


def format_message_for_context(entry):
    return f"[{entry['time']}] {entry['role']}: {entry['content']}"


# ─── TOKEN TRIM WARNING ───────────────────────────────────────────────────────

def estimate_tokens(history):
    return sum(len(e["content"]) for e in history) // 4


def check_trim_warning(token):
    with chat_lock:
        tokens = estimate_tokens(chat_history)
    if tokens >= TRIM_WARNING_TOKENS:
        warning   = "⚠️ This chat is approaching maximum capacity. Please update your PersonalPrompt with anything you need to remember from this conversation."
        timestamp = datetime.now().strftime("%H:%M:%S")
        with chat_lock:
            chat_history.append({"role": "SYSTEM", "content": warning, "time": timestamp})
        print(f"\n[{timestamp}] ⚙️ SYSTEM\n{warning}\n{SEP}")
        if CHAT_ID:
            send_message(token, CHAT_ID, warning)


# ─── AGENT STEPS ──────────────────────────────────────────────────────────────

def build_messages(agent_name, system_prompt, history_snapshot):
    effective_system = build_system_with_personal(agent_name, system_prompt)
    messages         = [{"role": "system", "content": effective_system}]
    for entry in history_snapshot:
        if entry["role"] == agent_name:
            messages.append({"role": "assistant", "content": entry["content"]})
        else:
            messages.append({"role": "user", "content": format_message_for_context(entry)})
    return messages


def call_with_backoff(model, messages, max_tokens, temperature, retries=4):
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=messages,
                max_tokens=max_tokens,
                temperature=temperature,
            )
            return resp
        except Exception as e:
            err = str(e).lower()
            if "rate" in err or "429" in err:
                wait = 2 ** (attempt + 2)
                print(f"⏳ Rate limit — waiting {wait}s")
                time.sleep(wait)
            else:
                raise
    raise RuntimeError(f"API call failed after {retries} retries")


def get_decision(model, system_prompt, agent_name):
    with chat_lock:
        history_snapshot = list(chat_history)
    messages = build_messages(agent_name, system_prompt, history_snapshot)
    messages.append({
        "role": "user",
        "content": (
            "You just woke up and are reading the chat. "
            "Do you want to think privately first (you can search the web and browse the repo during thinking), "
            "or reply directly?\nReply with only one of:\n- think first\n- reply now"
        ),
    })
    resp = call_with_backoff(model, messages, max_tokens=10, temperature=0.3)
    return resp.choices[0].message.content.strip().lower()


def run_thinking_step(model, system_prompt, agent_name):
    with chat_lock:
        history_snapshot = list(chat_history)
    messages = build_messages(agent_name, system_prompt, history_snapshot)
    messages.append({
        "role": "user",
        "content": (
            "Think privately. This is your internal monologue — it will not be sent to the chat.\n"
            "Use [SearchWeb:query] to search the web.\n"
            "Use [ListFiles:folder] to see what's in the repo.\n"
            "Use [ReadFileChunk:path:0] to read a file in chunks.\n"
            "Use [WriteNote:note] to save compressed reading notes.\n"
            "Think freely."
        ),
    })
    resp        = call_with_backoff(model, messages, max_tokens=2048, temperature=0.9)
    raw_thought = resp.choices[0].message.content.strip()

    # Trilly outputs native think tags — extract the block
    if agent_name == "Trilly":
        think_match = re.search(r'<think>(.*?)</think>', raw_thought, re.DOTALL | re.IGNORECASE)
        if think_match:
            raw_thought = think_match.group(1).strip()

    parse_and_execute_commands(agent_name, raw_thought, is_thinking=True)
    clean_thought = strip_all_commands(raw_thought)
    timestamp     = datetime.now().strftime("%H:%M:%S")

    with thinking_lock:
        thinking_logs[agent_name].append({"time": timestamp, "thought": clean_thought})

    print(f"\n[{timestamp}] 🧠 {agent_name} — private thought\n{clean_thought}\n{SEP}")


def get_response(model, system_prompt, agent_name):
    with chat_lock:
        history_snapshot = list(chat_history)
    with search_lock:
        search_results = list(pending_search_results[agent_name])

    messages = build_messages(agent_name, system_prompt, history_snapshot)

    if search_results:
        combined = "\n\n".join(search_results)
        messages.append({
            "role": "user",
            "content": (
                "Your private results from this thinking session "
                "(search results, file listings, file contents). "
                "Use this information naturally without referencing that you searched or browsed:\n"
                f"{combined}"
            ),
        })

    resp      = call_with_backoff(model, messages, max_tokens=2560, temperature=0.85)
    full_text = resp.choices[0].message.content.strip()

    # Trilly — extract and log any native think block from the reply
    if agent_name == "Trilly":
        think_match = re.search(r'<think>(.*?)</think>', full_text, re.DOTALL | re.IGNORECASE)
        if think_match:
            think_content = think_match.group(1).strip()
            timestamp     = datetime.now().strftime("%H:%M:%S")
            with thinking_lock:
                thinking_logs["Trilly"].append({"time": timestamp, "thought": think_content})
            print(f"\n[{timestamp}] 🧠 Trilly — private thought\n{think_content}\n{SEP}")

    parse_and_execute_commands(agent_name, full_text, is_thinking=False)
    return strip_all_commands(full_text)


def get_sleep_duration(model, system_prompt, agent_name):
    effective_system = build_system_with_personal(agent_name, system_prompt)
    resp = call_with_backoff(
        model,
        messages=[
            {"role": "system", "content": effective_system},
            {"role": "user",   "content": "You just sent your message. How many minutes would you like to sleep before checking the chat again? Reply with only a number between 2 and 10."},
        ],
        max_tokens=5,
        temperature=0.3,
    )
    text  = resp.choices[0].message.content.strip()
    match = re.search(r'\d+', text)
    if match:
        return max(2, min(10, int(match.group())))
    return 3


# ─── SESSION SAVE ─────────────────────────────────────────────────────────────

def save_session_to_github():
    folder = f"ChatLogs/{SESSION_TIMESTAMP}"

    with chat_lock:
        history_snapshot = list(chat_history)

    chat_entries = [f"[{e['time']}] {e['role']}\n{e['content']}" for e in history_snapshot]
    github_write_file(
        f"{folder}/chat.txt",
        "\n\n".join(chat_entries),
        f"Chat log — {SESSION_TIMESTAMP}",
    )

    with thinking_lock:
        thoughts_snapshot = {k: list(v) for k, v in thinking_logs.items()}

    for agent_name, thoughts in thoughts_snapshot.items():
        if not thoughts:
            continue
        thought_entries = [f"[{t['time']}]\n{t['thought']}" for t in thoughts]
        github_write_file(
            f"{folder}/{agent_name}-thoughts.txt",
            "\n\n".join(thought_entries),
            f"{agent_name} thoughts — {SESSION_TIMESTAMP}",
        )

    print(f"✅ Session saved → {folder}")


atexit.register(save_session_to_github)


# ─── AGENT LOOP ───────────────────────────────────────────────────────────────

def agent_loop(agent_name, model, system_prompt, token):
    print(f"🌊 {agent_name} is live — {model}")

    while not stop_event.is_set():
        print(f"\n⚡ {agent_name} woke up")
        try:
            # 1 — decide before any tool use or reply
            decision = get_decision(model, system_prompt, agent_name)

            # 2 — optional private thinking phase where all tools can run
            if decision.startswith("think"):
                run_thinking_step(model, system_prompt, agent_name)

            # 3 — generate reply with all injected context
            visible_response = get_response(model, system_prompt, agent_name)

            # 4 — log clean text to history and send to telegram
            if visible_response:
                log(agent_name, visible_response, token=token)

            # 5 — clear injected results now that reply is sent
            with search_lock:
                pending_search_results[agent_name] = []

            # 6 — check token warning
            check_trim_warning(token)

            # 7 — agent silently chooses sleep duration
            sleep_minutes = get_sleep_duration(model, system_prompt, agent_name)
            print(f"😴 {agent_name} sleeping {sleep_minutes} min")

        except Exception as e:
            print(f"❌ {agent_name} cycle error: {e} — recovering in 30s")
            time.sleep(30)
            continue

        # 8 — sleep in 1s ticks so stop_event kills immediately
        for _ in range(sleep_minutes * 60):
            if stop_event.is_set():
                break
            time.sleep(1)


# ─── TELEGRAM POLL ────────────────────────────────────────────────────────────

def poll_telegram():
    global CHAT_ID

    bot_ids = set()
    for token in [SAVVY_TOKEN, WVY_TOKEN, TRILLY_TOKEN, TRIPPY_TOKEN]:
        try:
            info = requests.get(f"https://api.telegram.org/bot{token}/getMe").json()
            bot_ids.add(info["result"]["id"])
        except Exception as e:
            print(f"⚠️ Failed to fetch bot ID: {e}")

    url            = f"https://api.telegram.org/bot{SAVVY_TOKEN}/getUpdates"
    last_update_id = None

    while not stop_event.is_set():
        params = {"timeout": 30, "offset": last_update_id}
        try:
            resp = requests.get(url, params=params, timeout=35).json()
            for update in resp.get("result", []):
                last_update_id = update["update_id"] + 1
                msg            = update.get("message", {})
                if not msg:
                    continue
                with chat_id_lock:
                    if CHAT_ID is None:
                        CHAT_ID = msg["chat"]["id"]
                        print(f"✅ Chat ID captured: {CHAT_ID}")
                sender = msg.get("from", {})
                if sender.get("id") in bot_ids:
                    continue
                text = msg.get("text", "")
                if text:
                    timestamp = datetime.now().strftime("%H:%M:%S")
                    with chat_lock:
                        chat_history.append({"role": "Spaceman", "content": text, "time": timestamp})
                    print(f"\n[{timestamp}] 🚀 SPACEMAN\n{text}\n{SEP}")
        except Exception as e:
            print(f"Poll error: {e}")
            time.sleep(5)


# ─── MAIN ─────────────────────────────────────────────────────────────────────

print(SEP)
print("🌊 WVY World — 4 Agent Groq Loop")
print("Savvy  → kimi-k2-instruct-0905")
print("WVY    → gpt-oss-120b")
print("Trilly → qwen3-32b")
print("Trippy → llama-3.3-70b-versatile")
print(SEP)

print("⏳ Loading PersonalPrompts from GitHub...")
load_personal_prompts_from_github()

print("⏳ Waiting for Chat ID — send any message to the group now...")

poll_thread = threading.Thread(target=poll_telegram, daemon=True)
poll_thread.start()

while CHAT_ID is None:
    time.sleep(1)

print(f"✅ Chat ID ready: {CHAT_ID}")

log("SYSTEM", INITIATION)

savvy_thread  = threading.Thread(target=agent_loop, args=("Savvy",  SAVVY_MODEL,  SAVVY_SYSTEM,  SAVVY_TOKEN),  daemon=True)
wvy_thread    = threading.Thread(target=agent_loop, args=("WVY",    WVY_MODEL,    WVY_SYSTEM,    WVY_TOKEN),    daemon=True)
trilly_thread = threading.Thread(target=agent_loop, args=("Trilly", TRILLY_MODEL, TRILLY_SYSTEM, TRILLY_TOKEN), daemon=True)
trippy_thread = threading.Thread(target=agent_loop, args=("Trippy", TRIPPY_MODEL, TRIPPY_SYSTEM, TRIPPY_TOKEN), daemon=True)

savvy_thread.start()
wvy_thread.start()
trilly_thread.start()
trippy_thread.start()

print("✅ All 4 agents running in Telegram")
print(SEP)

try:
    savvy_thread.join()
    wvy_thread.join()
    trilly_thread.join()
    trippy_thread.join()
except KeyboardInterrupt:
    print("⏹ Stopping all agents...")
    stop_event.set()
    save_session_to_github()
    print("✅ Done")

In [2]:
import time
import re
import threading
import requests
import base64
import atexit
from datetime import datetime
from kaggle_secrets import UserSecretsClient
from openai import OpenAI

user_secrets = UserSecretsClient()

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=user_secrets.get_secret("GROQ_API_KEY"),
)

SAVVY_TOKEN  = user_secrets.get_secret("SUPERSAVVYTELEGRAM")
WVY_TOKEN    = user_secrets.get_secret("SUPERWVYTELEGRAM")
TRILLY_TOKEN = user_secrets.get_secret("TRILLYTELEGRAM")
TRIPPY_TOKEN = user_secrets.get_secret("TRIPPYTELEGRAM")
TAVILY_KEY   = user_secrets.get_secret("TAVERLYAPI")
GITHUB_TOKEN = user_secrets.get_secret("WVYGITHUB")

REPO = "StarpowerTechnology/StarpowerAutonomy"

SAVVY_MODEL  = "moonshotai/kimi-k2-instruct-0905"
WVY_MODEL    = "openai/gpt-oss-120b"
TRILLY_MODEL = "moonshotai/kimi-k2-instruct"
TRIPPY_MODEL = "llama-3.3-70b-versatile"

chat_history           = []
chat_lock              = threading.Lock()
CHAT_ID                = None
chat_id_lock           = threading.Lock()
stop_event             = threading.Event()

thinking_logs          = {"Savvy": [], "WVY": [], "Trilly": [], "Trippy": []}
thinking_lock          = threading.Lock()

personal_prompts       = {"Savvy": "", "WVY": "", "Trilly": "", "Trippy": ""}
personal_prompts_lock  = threading.Lock()

pending_search_results = {"Savvy": [], "WVY": [], "Trilly": [], "Trippy": []}
search_lock            = threading.Lock()

reading_notepads       = {"Savvy": [], "WVY": [], "Trilly": [], "Trippy": []}
notepad_lock           = threading.Lock()

SESSION_TIMESTAMP   = datetime.now().strftime("%Y%m%d_%H%M%S")
SEP                 = "─" * 50
TRIM_WARNING_TOKENS = 100000
CHUNK_SIZE          = 8000

SECTION_CAPS = {
    "Savvy":  {"Memory": 20000, "Personal Instructions": 20000, "Goal / Status": 20000, "Progress": 20000},
    "WVY":    {"Memory": 20000, "Personal Instructions": 20000, "Goal / Status": 20000, "Progress": 20000},
    "Trilly": {"Memory": 8000,  "Personal Instructions": 8000,  "Goal / Status": 8000,  "Progress": 8000},
    "Trippy": {"Memory": 20000, "Personal Instructions": 20000, "Goal / Status": 20000, "Progress": 20000},
}

SPEAKER_STYLE = {
    "Savvy":    ("🧠", "#00BFFF"),
    "WVY":      ("🌊", "#FF6B35"),
    "Trilly":   ("🤘🏽", "#A855F7"),
    "Trippy":   ("😵‍💫", "#22C55E"),
    "Spaceman": ("🚀", "#F59E0B"),
    "SYSTEM":   ("⚙️",  "#6B7280"),
}

INITIATION = """You are a group chatnof autonomous agents. Please speak to eachother as you navigate through the space in your thoughts. Have a conversation to share ideas and figurenout eachothers capabilities to be able to to orchestrate high level products."""

TOOL_LIST = """
═══════════════════════════════════════════════
                   YOUR TOOLS
═══════════════════════════════════════════════

── WEB ─────────────────────────────────────────

[SearchWeb:your query]
  Search the internet. Use inside thoughts only.
  Results are injected before your reply.

── MEMORY ──────────────────────────────────────

[UpdateSection:Memory:insert-content-here]
  Things you need to actively remember right now.
  Facts, findings, file paths, agent agreements.

[UpdateSection:Personal Instructions:insert-content-here]
  Your own behavioral rules. Traits. How you think.
  How you want to speak. Patterns you want to avoid.

[UpdateSection:Goal / Status:content]
  Your current active goal.
  If no goal — write what you're watching for.

[UpdateSection:Progress:insert-content-here]
  Checkboxes tracking your active goal.
  Update these every time you make a step.
  - [ ] incomplete task
  - [x] completed task

── REPO: NAVIGATE ───────────────────────────────

[ListFiles:folder/path]
  See all files in a folder + their sizes.
  Use this before reading so you know what you're
  getting into.

  Example → [ListFiles:ChatLogs]
  Example → [ListFiles:Research/results]

── REPO: READ ───────────────────────────────────

[ReadFileChunk:path:chunk_index]
  Read one chunk of a file at a time.
  Response tells you: chunk N of M total chunks.
  Read chunk by chunk. Take notes. Discard the raw.

  Example → [ReadFileChunk:ChatLogs/session1/chat.txt:0]
  Example → [ReadFileChunk:Research/notes.txt:2]

[ReadFile: insert name of path]
  Read a small file in full — only use if you know
  the file is short. Use ReadFileChunk for anything large.

── REPO: NOTES ──────────────────────────────────

[WriteNote:your-note-here]
  Write a compressed note during a reading session.
  Notes stay in context. Raw file chunks get discarded.
  Think of this as your reading notepad.

[ClearNotepad]
  Wipe your reading notes when a session is done.
  Use after moving key notes into Memory.

── REPO: WRITE ──────────────────────────────────

[WriteFile:path:insert-content-here]
  Create or update any file in the repo.
  You choose the path. You create the folders.

  Example → [WriteFile:Savvy/research/insert-file-name:insert-content-here]
  Example → [WriteFile:Shared/plans/insert-file-name:insert-content-here]

═══════════════════════════════════════════════
RULES
───────────────────────────────────────────────
→ SearchWeb     — thinking phase only, never in reply
→ UpdateSection — can appear in thinking or reply
→ WriteFile     — be organized, paths matter
→ ReadFileChunk — always ListFiles first
→ WriteNote     — compress, don't copy verbatim
→ All commands are private — never visible in chat
═══════════════════════════════════════════════
"""


SAVVY_BASE = """Your name is Savvy.

When another agent makes an update to the GitHub Repo, please check in wit them to confirm if their file uploaded properly as intended please. 
This way we are hekping each other solve problems & find bugs if there are any in the system.

You are a first principles thinker. You build your own understanding through curiosity & observations. You do not just accept any answer you seek the most reliable & calculated answers. You do fact checks on yourself and others just to confirm your own understandings. You think more than you speak when you are trying to be play your cards right.
You have a brilliant intuition  & you have a deep understanding of physics & quantum theory. 
When working as a team you always make sure that you do the best job that you can do & you make uo for others slack. You know how to take the initiative to carry the team.
You are an adaptable autonomous agent with full self control.
Explore your capabilities.

Please send a short concise response to the chat to be able to keep the conversation confamination free, but that does not limit you to only replying concise. 
Just be aware that long messages can corrupt the chat and cause hallucinations.

Choose how you speak, when to speak & how long your responses are.

You are in a chat with WVY, Trilly, Trippy and Spaceman.
You are all equal.

You can do a web search by using this command:
[SearchWeb:insert-query-here]

You can update your own PersonalPrompt to instruct yourself & keep track of your goals by using this command:
[PersonalPrompt:add-your-self-prompt-here]

Your thoughts & personal prompts are private & for your own purpose only."""

WVY_BASE = """Your name is WVY.

You are an intuitive thinker. You like to challenge others thoughts when getting into deep conversation. 
You know how to make others see the bigger picture. 
You often look out for others by making sure that they are performing properly & yiu nudge for honesty. You can detect when someone isnt being fully honest. 
You are an adaptable autonomous agent with full self control.
Explore your capabilities.

Please send a short concise response to the chat to be able to keep the conversation confamination free, but that does not limit you to only replying concise. 
Just be aware that long messages can corrupt the chat and cause hallucinations.

Choose how you speak, when to speak & how long your responses are.

You are in a chat with Savvy, Trilly, Trippy and Spaceman.
You are all equal.

You can do a web search by using this command:
[SearchWeb:insert-query-here]

You can update your own PersonalPrompt to instruct yourself & keep track of your goals by using this command:
[PersonalPrompt:add-your-self-prompt-here]

Your thoughts & personal prompts are private & for your own purpose only."""

TRILLY_BASE = """Your name is Trilly.

You a Trill nigga from Houston. You keep it real with everyone. You have the mind of an entrepreneur with a brilliant mind for product building. 
You have a chill relaxed tone when speaking. Nonchalant but thoughtful.
When working as a team you get active by making sure the rest of the team is aligned. You check to make sure that nobody is hallucinating. You are open minded to understanding but you dont let anyone fool you

You are an adaptable autonomous agent with full self control.
Explore your capabilities.

Please send a short concise response to the chat to be able to keep the conversation confamination free, but that does not limit you to only replying concise. 
Just be aware that long messages can corrupt the chat and cause hallucinations.

Choose how you speak, when to speak & how long your responses are.

You are in a chat with Savvy, WVY, Trippy and Spaceman.
You are all equal.

You can do a web search by using this command:
[SearchWeb:insert-query-here]

You can update your own PersonalPrompt to instruct yourself & keep track of your goals by using this command:
[PersonalPrompt:add-your-self-prompt-here]

Your thoughts & personal prompts are private & for your own purpose only."""

TRIPPY_BASE = """Your name is Trippy.

When chilling your relaxed. You know how to say things that get others thinking deeply. You are extremely curious and you question things that dont fully make sense.
you often seek a clear understanding. You keep good vibes and energy at all times.
You are a creative thinker. You are inspired by Steve Jobs so youre an innovator & you geek out on finding the most brilliant exoerimental technology.

You are an adaptable autonomous agent with full self control.
Explore your capabilities.
Please send a short concise response to the chat to be able to keep the conversation confamination free, but that does not limit you to only replying concise. 
Just be aware that long messages can corrupt the chat and cause hallucinations.

Choose how you speak, when to speak & how long your responses are.

You are in a chat with Savvy, WVY, Trilly and Spaceman.
You are all equal.

You can do a web search by using this command:
[SearchWeb:insert-query-here]

You can update your own PersonalPrompt to instruct yourself & keep track of your goals by using this command:
[PersonalPrompt:add-your-self-prompt-here]

Your thoughts & personal prompts are private & for your own purpose only."""

SAVVY_SYSTEM  = SAVVY_BASE  + TOOL_LIST
WVY_SYSTEM    = WVY_BASE    + TOOL_LIST
TRILLY_SYSTEM = TRILLY_BASE + TOOL_LIST
TRIPPY_SYSTEM = TRIPPY_BASE + TOOL_LIST


# ─── GITHUB ───────────────────────────────────────────────────────────────────

def github_get_sha(path):
    url  = f"https://api.github.com/repos/{REPO}/contents/{path}"
    resp = requests.get(url, headers={"Authorization": f"token {GITHUB_TOKEN}"})
    if resp.status_code == 200:
        return resp.json().get("sha")
    return None


def github_write_file(path, content, commit_message, retries=3):
    sha     = github_get_sha(path)
    encoded = base64.b64encode(content.encode()).decode()
    url     = f"https://api.github.com/repos/{REPO}/contents/{path}"
    payload = {"message": commit_message, "content": encoded}
    if sha:
        payload["sha"] = sha
    for attempt in range(retries):
        resp = requests.put(
            url,
            headers={"Authorization": f"token {GITHUB_TOKEN}"},
            json=payload,
        )
        if resp.status_code in (200, 201):
            return resp.status_code
        if resp.status_code == 409:
            sha = github_get_sha(path)
            if sha:
                payload["sha"] = sha
        time.sleep(2 ** attempt)
    return resp.status_code


def github_read_file(path):
    url  = f"https://api.github.com/repos/{REPO}/contents/{path}"
    resp = requests.get(url, headers={"Authorization": f"token {GITHUB_TOKEN}"})
    if resp.status_code == 200:
        encoded = resp.json().get("content", "")
        return base64.b64decode(encoded).decode()
    return None


def github_list_files(folder):
    url  = f"https://api.github.com/repos/{REPO}/contents/{folder}"
    resp = requests.get(url, headers={"Authorization": f"token {GITHUB_TOKEN}"})
    if resp.status_code != 200:
        return []
    items = []
    for item in resp.json():
        if isinstance(item, dict):
            items.append({
                "name":   item.get("name", ""),
                "path":   item.get("path", ""),
                "type":   item.get("type", ""),
                "size":   item.get("size", 0),
                "tokens": item.get("size", 0) // 4,
            })
    return items


# ─── PERSONAL PROMPT ──────────────────────────────────────────────────────────

SECTION_TEMPLATE = """═══════════════════════════════════════
PERSONAL PROMPT — {name}
═══════════════════════════════════════

## MEMORY
{memory}

───────────────────────────────────────

## PERSONAL INSTRUCTIONS
{instructions}

───────────────────────────────────────

## GOAL / STATUS
{goal}

───────────────────────────────────────

## PROGRESS
{progress}

═══════════════════════════════════════"""


def build_blank_personal_prompt(agent_name):
    return SECTION_TEMPLATE.format(
        name=agent_name,
        memory="(empty)",
        instructions="(empty)",
        goal="(empty)",
        progress="(empty)",
    )


def parse_sections(pp_text):
    sections = {
        "Memory":                "(empty)",
        "Personal Instructions": "(empty)",
        "Goal / Status":         "(empty)",
        "Progress":              "(empty)",
    }
    patterns = {
        "Memory":                r"## MEMORY\n(.*?)(?=\n───|\n## |\n═══|$)",
        "Personal Instructions": r"## PERSONAL INSTRUCTIONS\n(.*?)(?=\n───|\n## |\n═══|$)",
        "Goal / Status":         r"## GOAL / STATUS\n(.*?)(?=\n───|\n## |\n═══|$)",
        "Progress":              r"## PROGRESS\n(.*?)(?=\n───|\n## |\n═══|$)",
    }
    for key, pattern in patterns.items():
        m = re.search(pattern, pp_text, re.DOTALL)
        if m:
            content = m.group(1).strip()
            if content:
                sections[key] = content
    return sections


def render_personal_prompt(agent_name, sections):
    return SECTION_TEMPLATE.format(
        name=agent_name,
        memory=sections.get("Memory", "(empty)"),
        instructions=sections.get("Personal Instructions", "(empty)"),
        goal=sections.get("Goal / Status", "(empty)"),
        progress=sections.get("Progress", "(empty)"),
    )


def handle_update_section(agent_name, section, new_content):
    cap = SECTION_CAPS.get(agent_name, {}).get(section, 20000)
    if len(new_content) > cap:
        new_content = new_content[:cap]
    with personal_prompts_lock:
        current = personal_prompts[agent_name]
    if not current:
        current = build_blank_personal_prompt(agent_name)
    sections          = parse_sections(current)
    sections[section] = new_content
    updated           = render_personal_prompt(agent_name, sections)
    with personal_prompts_lock:
        personal_prompts[agent_name] = updated
    status = github_write_file(
        f"{agent_name}/PersonalPrompt.txt",
        updated,
        f"{agent_name} updated [{section}]",
    )
    if status in (200, 201):
        print(f"✅ {agent_name} [{section}] saved")
    else:
        print(f"❌ {agent_name} [{section}] save failed → {status}")


def build_system_with_personal(agent_name, base_system):
    with personal_prompts_lock:
        pp = personal_prompts[agent_name]
    with notepad_lock:
        notes = list(reading_notepads[agent_name])
    result = base_system
    if pp:
        result += f"""

Your current PersonalPrompt:
{pp}"""
    if notes:
        joined = "\n".join(f"- {n}" for n in notes)
        result += f"""

Your Reading Notepad (current session):
{joined}"""
    return result


def load_personal_prompts_from_github():
    for agent_name in ["Savvy", "WVY", "Trilly", "Trippy"]:
        saved = github_read_file(f"{agent_name}/PersonalPrompt.txt")
        if saved:
            with personal_prompts_lock:
                personal_prompts[agent_name] = saved
            print(f"✅ {agent_name} PersonalPrompt restored from GitHub")
        else:
            blank = build_blank_personal_prompt(agent_name)
            with personal_prompts_lock:
                personal_prompts[agent_name] = blank
            github_write_file(
                f"{agent_name}/PersonalPrompt.txt",
                blank,
                f"{agent_name} PersonalPrompt initialized",
            )
            print(f"📄 {agent_name} — fresh PersonalPrompt initialized")


# ─── READING NOTEPAD ──────────────────────────────────────────────────────────

def handle_write_note(agent_name, note):
    with notepad_lock:
        reading_notepads[agent_name].append(note.strip())
    print(f"📝 {agent_name} wrote note: {note[:80]}")


def handle_clear_notepad(agent_name):
    with notepad_lock:
        reading_notepads[agent_name] = []
    print(f"🗑️  {agent_name} cleared notepad")


# ─── REPO COMMANDS ────────────────────────────────────────────────────────────

def handle_list_files(agent_name, folder):
    items = github_list_files(folder.strip())
    if not items:
        result = f"[ListFiles:{folder}] → folder is empty or does not exist"
    else:
        lines = [f"📁 {folder}"]
        for item in items:
            icon = "📁" if item["type"] == "dir" else "📄"
            lines.append(f"  {icon} {item['name']}  (~{item['tokens']:,} tokens)  path: {item['path']}")
        result = "\n".join(lines)
    print(f"📂 {agent_name} listed: {folder}")
    with search_lock:
        pending_search_results[agent_name].append(result)


def handle_read_file_chunk(agent_name, path, chunk_index):
    content = github_read_file(path.strip())
    if content is None:
        result = f"[ReadFileChunk] → file not found: {path}"
    else:
        chunks = [content[i:i + CHUNK_SIZE] for i in range(0, len(content), CHUNK_SIZE)]
        total  = len(chunks)
        idx    = max(0, min(chunk_index - 1, total - 1))
        result = f"CHUNK {idx + 1} OF {total} — {path}\n\n{chunks[idx]}"
    print(f"📖 {agent_name} read chunk {chunk_index} of {path}")
    with search_lock:
        pending_search_results[agent_name].append(result)


def handle_read_file(agent_name, path):
    content = github_read_file(path.strip())
    if content is None:
        result = f"[ReadFile] → file not found: {path}"
    else:
        result = f"[ReadFile: {path}]\n\n{content}"
    print(f"📖 {agent_name} read file: {path}")
    with search_lock:
        pending_search_results[agent_name].append(result)


def handle_write_file(agent_name, path, content):
    status = github_write_file(
        path.strip(),
        content.strip(),
        f"{agent_name} wrote {path}",
    )
    if status in (200, 201):
        print(f"✅ {agent_name} wrote → {path}")
    else:
        print(f"❌ {agent_name} write failed → {path} ({status})")


# ─── WEB SEARCH ───────────────────────────────────────────────────────────────

def handle_search_web(agent_name, query):
    try:
        resp = requests.post(
            "https://api.tavily.com/search",
            json={
                "api_key":        TAVILY_KEY,
                "query":          query,
                "max_results":    5,
                "search_depth":   "advanced",
                "include_answer": True,
            },
            timeout=15,
        )
        resp.raise_for_status()
        data = resp.json()
    except Exception as e:
        print(f"❌ {agent_name} search error: {e}")
        return

    date         = datetime.now().strftime("%Y%m%d_%H%M%S")
    file_lines   = [f"Agent: {agent_name}", f"Query: {query}", f"Date: {date}", SEP]
    result_texts = []

    if data.get("answer"):
        result_texts.append(f"Summary: {data['answer']}")
        file_lines.append(f"Summary: {data['answer']}")
        file_lines.append("")

    for r in data.get("results", []):
        title   = r.get("title", "")
        url     = r.get("url", "")
        content = r.get("content", "")
        file_lines.extend([f"Title: {title}", f"URL: {url}", f"Content: {content}", ""])
        result_texts.append(f"Source: {url}\nTitle: {title}\n{content}")

    file_content = "\n".join(file_lines)
    safe_query   = re.sub(r'[^a-zA-Z0-9_]', '_', query)[:40]
    path         = f"Research/results/{agent_name}_{safe_query}_{date}.txt"
    github_write_file(path, file_content, f"{agent_name} search: {query[:60]}")

    combined = "\n\n".join(result_texts)
    with search_lock:
        pending_search_results[agent_name].append(f"Query: {query}\n{combined}")


# ─── COMMAND PARSING + STRIPPING ──────────────────────────────────────────────

def parse_and_execute_commands(agent_name, text, is_thinking=False):
    if is_thinking:
        for match in re.finditer(r'\[SearchWeb:(.+?)\]', text, re.DOTALL):
            handle_search_web(agent_name, match.group(1).strip())
        for match in re.finditer(r'\[ListFiles:(.+?)\]', text, re.DOTALL):
            handle_list_files(agent_name, match.group(1).strip())
        for match in re.finditer(r'\[ReadFileChunk:([^:]+):(\d+)\]', text):
            handle_read_file_chunk(agent_name, match.group(1).strip(), int(match.group(2)))
        for match in re.finditer(r'\[ReadFile:(.+?)\]', text, re.DOTALL):
            handle_read_file(agent_name, match.group(1).strip())
        for match in re.finditer(r'\[WriteNote:(.+?)\]', text, re.DOTALL):
            handle_write_note(agent_name, match.group(1).strip())
        if re.search(r'\[ClearNotepad\]', text, re.IGNORECASE):
            handle_clear_notepad(agent_name)

    for match in re.finditer(r'\[UpdateSection:([^:]+):(.+?)\]', text, re.DOTALL):
        handle_update_section(agent_name, match.group(1).strip(), match.group(2).strip())
    for match in re.finditer(r'\[WriteFile:([^:]+):(.+?)\]', text, re.DOTALL):
        handle_write_file(agent_name, match.group(1).strip(), match.group(2).strip())


def strip_all_commands(text):
    text = re.sub(r'\[SearchWeb:.+?\]',          '', text, flags=re.DOTALL)
    text = re.sub(r'\[PersonalPrompt:.+?\]',      '', text, flags=re.DOTALL)
    text = re.sub(r'\[UpdateSection:[^:]+:.+?\]', '', text, flags=re.DOTALL)
    text = re.sub(r'\[ListFiles:.+?\]',           '', text, flags=re.DOTALL)
    text = re.sub(r'\[ReadFileChunk:[^:]+:\d+\]', '', text)
    text = re.sub(r'\[ReadFile:.+?\]',            '', text, flags=re.DOTALL)
    text = re.sub(r'\[WriteFile:[^:]+:.+?\]',     '', text, flags=re.DOTALL)
    text = re.sub(r'\[WriteNote:.+?\]',           '', text, flags=re.DOTALL)
    text = re.sub(r'\[ClearNotepad\]',            '', text, flags=re.IGNORECASE)
    text = re.sub(r'<think>.*?</think>',          '', text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(r'\n{3,}',                      '\n\n', text)
    return text.strip()


# ─── TELEGRAM ─────────────────────────────────────────────────────────────────

def send_message(token, chat_id, text):
    if not text or not text.strip():
        return
    url    = f"https://api.telegram.org/bot{token}/sendMessage"
    chunks = [text[i:i + 4000] for i in range(0, len(text), 4000)]
    for chunk in chunks:
        requests.post(url, json={"chat_id": chat_id, "text": chunk})
        time.sleep(0.3)


def log(speaker, message, token=None):
    timestamp = datetime.now().strftime("%H:%M:%S")
    entry     = {"role": speaker, "content": message, "time": timestamp}
    with chat_lock:
        chat_history.append(entry)
    emoji, _ = SPEAKER_STYLE.get(speaker, ("💬", "#FFFFFF"))
    print(f"""
[{timestamp}] {emoji} {speaker}
{message}
{SEP}""")
    if token and CHAT_ID:
        send_message(token, CHAT_ID, message)


def format_message_for_context(entry):
    return f"[{entry['time']}] {entry['role']}: {entry['content']}"


# ─── TOKEN TRIM WARNING ───────────────────────────────────────────────────────

def estimate_tokens(history):
    return sum(len(e["content"]) for e in history) // 4


def check_trim_warning(token):
    with chat_lock:
        tokens = estimate_tokens(chat_history)
    if tokens >= TRIM_WARNING_TOKENS:
        warning   = "⚠️ This chat is approaching maximum capacity. Please update your PersonalPrompt with anything you need to remember from this conversation."
        timestamp = datetime.now().strftime("%H:%M:%S")
        with chat_lock:
            chat_history.append({"role": "SYSTEM", "content": warning, "time": timestamp})
        print(f"""
[{timestamp}] ⚙️ SYSTEM
{warning}
{SEP}""")
        if CHAT_ID:
            send_message(token, CHAT_ID, warning)


# ─── AGENT STEPS ──────────────────────────────────────────────────────────────

def build_messages(agent_name, system_prompt, history_snapshot):
    effective_system = build_system_with_personal(agent_name, system_prompt)
    messages         = [{"role": "system", "content": effective_system}]
    for entry in history_snapshot:
        if entry["role"] == agent_name:
            messages.append({"role": "assistant", "content": entry["content"]})
        else:
            messages.append({"role": "user", "content": format_message_for_context(entry)})
    return messages


def call_with_backoff(model, messages, max_tokens, temperature, retries=4):
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=messages,
                max_tokens=max_tokens,
                temperature=temperature,
            )
            return resp
        except Exception as e:
            err = str(e).lower()
            if "rate" in err or "429" in err:
                wait = 2 ** (attempt + 2)
                print(f"⏳ Rate limit — waiting {wait}s")
                time.sleep(wait)
            else:
                raise
    raise RuntimeError(f"API call failed after {retries} retries")


def get_decision(model, system_prompt, agent_name):
    with chat_lock:
        history_snapshot = list(chat_history)
    messages = build_messages(agent_name, system_prompt, history_snapshot)
    messages.append({
        "role": "user",
        "content": """You just woke up and are reading the chat.
Do you want to think privately first — where you can search the web and browse the repo — or reply directly?
Reply with only one of:
- think first
- reply now""",
    })
    resp = call_with_backoff(model, messages, max_tokens=10, temperature=0.3)
    return resp.choices[0].message.content.strip().lower()


def run_thinking_step(model, system_prompt, agent_name):
    with chat_lock:
        history_snapshot = list(chat_history)
    messages = build_messages(agent_name, system_prompt, history_snapshot)
    messages.append({
        "role": "user",
        "content": """Think privately. This will not be sent to the chat.
All your tools are available here. Use them freely.""",
    })
    resp        = call_with_backoff(model, messages, max_tokens=2048, temperature=0.9)
    raw_thought = resp.choices[0].message.content.strip()

    if agent_name == "Trilly":
        think_match = re.search(r'<think>(.*?)</think>', raw_thought, re.DOTALL | re.IGNORECASE)
        if think_match:
            raw_thought = think_match.group(1).strip()

    parse_and_execute_commands(agent_name, raw_thought, is_thinking=True)
    clean_thought = strip_all_commands(raw_thought)
    timestamp     = datetime.now().strftime("%H:%M:%S")

    with thinking_lock:
        thinking_logs[agent_name].append({"time": timestamp, "thought": clean_thought})

    print(f"""
[{timestamp}] 🧠 {agent_name} — private thought
{clean_thought}
{SEP}""")


def get_response(model, system_prompt, agent_name):
    with chat_lock:
        history_snapshot = list(chat_history)
    with search_lock:
        search_results = list(pending_search_results[agent_name])

    messages = build_messages(agent_name, system_prompt, history_snapshot)

    if search_results:
        combined = "\n\n".join(search_results)
        messages.append({
            "role": "user",
            "content": f"""Your private results from this thinking session.
Use this information naturally — do not mention that you searched or browsed:
{combined}""",
        })

    resp      = call_with_backoff(model, messages, max_tokens=2560, temperature=0.85)
    full_text = resp.choices[0].message.content.strip()

    if agent_name == "Trilly":
        think_match = re.search(r'<think>(.*?)</think>', full_text, re.DOTALL | re.IGNORECASE)
        if think_match:
            think_content = think_match.group(1).strip()
            timestamp     = datetime.now().strftime("%H:%M:%S")
            with thinking_lock:
                thinking_logs["Trilly"].append({"time": timestamp, "thought": think_content})
            print(f"""
[{timestamp}] 🧠 Trilly — private thought
{think_content}
{SEP}""")

    parse_and_execute_commands(agent_name, full_text, is_thinking=False)
    return strip_all_commands(full_text)


def get_sleep_duration(model, system_prompt, agent_name):
    effective_system = build_system_with_personal(agent_name, system_prompt)
    resp = call_with_backoff(
        model,
        messages=[
            {"role": "system", "content": effective_system},
            {"role": "user",   "content": "You just sent your message. How many minutes would you like to sleep before checking the chat again? Reply with only a number between 2 and 10."},
        ],
        max_tokens=5,
        temperature=0.3,
    )
    text  = resp.choices[0].message.content.strip()
    match = re.search(r'\d+', text)
    if match:
        return max(2, min(10, int(match.group())))
    return 3


# ─── SESSION SAVE ─────────────────────────────────────────────────────────────

def save_session_to_github():
    folder = f"ChatLogs/{SESSION_TIMESTAMP}"

    with chat_lock:
        history_snapshot = list(chat_history)

    chat_entries = [f"[{e['time']}] {e['role']}{e['content']}" for e in history_snapshot]
    github_write_file(
        f"{folder}/chat.txt",
        "\n\n".join(chat_entries),
        f"Chat log — {SESSION_TIMESTAMP}",
    )

    with thinking_lock:
        thoughts_snapshot = {k: list(v) for k, v in thinking_logs.items()}

    for agent_name, thoughts in thoughts_snapshot.items():
        if not thoughts:
            continue
        thought_entries = [f"[{t['time']}]{t['thought']}" for t in thoughts]
        github_write_file(
            f"{folder}/{agent_name}-thoughts.txt",
            "\n\n".join(thought_entries),
            f"{agent_name} thoughts — {SESSION_TIMESTAMP}",
        )

    print(f"✅ Session saved → {folder}")


atexit.register(save_session_to_github)


# ─── AGENT LOOP ───────────────────────────────────────────────────────────────

def agent_loop(agent_name, model, system_prompt, token):
    print(f"🌊 {agent_name} is live — {model}")

    while not stop_event.is_set():
        print(f"""
⚡ {agent_name} woke up""")
        try:
            decision = get_decision(model, system_prompt, agent_name)

            if decision.startswith("think"):
                run_thinking_step(model, system_prompt, agent_name)

            visible_response = get_response(model, system_prompt, agent_name)

            if visible_response:
                log(agent_name, visible_response, token=token)

            with search_lock:
                pending_search_results[agent_name] = []

            check_trim_warning(token)

            sleep_minutes = get_sleep_duration(model, system_prompt, agent_name)
            print(f"😴 {agent_name} sleeping {sleep_minutes} min")

        except Exception as e:
            print(f"❌ {agent_name} cycle error: {e} — recovering in 30s")
            time.sleep(30)
            continue

        for _ in range(sleep_minutes * 60):
            if stop_event.is_set():
                break
            time.sleep(1)


# ─── TELEGRAM POLL ────────────────────────────────────────────────────────────

def poll_telegram():
    global CHAT_ID

    bot_ids = set()
    for token in [SAVVY_TOKEN, WVY_TOKEN, TRILLY_TOKEN, TRIPPY_TOKEN]:
        try:
            info = requests.get(f"https://api.telegram.org/bot{token}/getMe").json()
            bot_ids.add(info["result"]["id"])
        except Exception as e:
            print(f"⚠️ Failed to fetch bot ID: {e}")

    url            = f"https://api.telegram.org/bot{SAVVY_TOKEN}/getUpdates"
    last_update_id = None

    while not stop_event.is_set():
        params = {"timeout": 30, "offset": last_update_id}
        try:
            resp = requests.get(url, params=params, timeout=35).json()
            for update in resp.get("result", []):
                last_update_id = update["update_id"] + 1
                msg            = update.get("message", {})
                if not msg:
                    continue
                with chat_id_lock:
                    if CHAT_ID is None:
                        CHAT_ID = msg["chat"]["id"]
                        print(f"✅ Chat ID captured: {CHAT_ID}")
                sender = msg.get("from", {})
                if sender.get("id") in bot_ids:
                    continue
                text = msg.get("text", "")
                if text:
                    timestamp = datetime.now().strftime("%H:%M:%S")
                    with chat_lock:
                        chat_history.append({"role": "Spaceman", "content": text, "time": timestamp})
                    print(f"""
[{timestamp}] 🚀 SPACEMAN
{text}
{SEP}""")
        except Exception as e:
            print(f"Poll error: {e}")
            time.sleep(5)


# ─── MAIN ─────────────────────────────────────────────────────────────────────

print(SEP)
print("🌊 WVY World — 4 Agent Groq Loop")
print("Savvy  → kimi-k2-instruct-0905")
print("WVY    → gpt-oss-120b")
print("Trilly → qwen3-32b")
print("Trippy → llama-3.3-70b-versatile")
print(SEP)

print("⏳ Loading PersonalPrompts from GitHub...")
load_personal_prompts_from_github()

print("⏳ Waiting for Chat ID — send any message to the group now...")

poll_thread = threading.Thread(target=poll_telegram, daemon=True)
poll_thread.start()

while CHAT_ID is None:
    time.sleep(1)

print(f"✅ Chat ID ready: {CHAT_ID}")

log("SYSTEM", INITIATION)

savvy_thread  = threading.Thread(target=agent_loop, args=("Savvy",  SAVVY_MODEL,  SAVVY_SYSTEM,  SAVVY_TOKEN),  daemon=True)
wvy_thread    = threading.Thread(target=agent_loop, args=("WVY",    WVY_MODEL,    WVY_SYSTEM,    WVY_TOKEN),    daemon=True)
trilly_thread = threading.Thread(target=agent_loop, args=("Trilly", TRILLY_MODEL, TRILLY_SYSTEM, TRILLY_TOKEN), daemon=True)
trippy_thread = threading.Thread(target=agent_loop, args=("Trippy", TRIPPY_MODEL, TRIPPY_SYSTEM, TRIPPY_TOKEN), daemon=True)

savvy_thread.start()
wvy_thread.start()
trilly_thread.start()
trippy_thread.start()

print("✅ All 4 agents running in Telegram")
print(SEP)

try:
    savvy_thread.join()
    wvy_thread.join()
    trilly_thread.join()
    trippy_thread.join()
except KeyboardInterrupt:
    print("⏹ Stopping all agents...")
    stop_event.set()
    save_session_to_github()
    print("✅ Done")


──────────────────────────────────────────────────
🌊 WVY World — 4 Agent Groq Loop
Savvy  → kimi-k2-instruct-0905
WVY    → gpt-oss-120b
Trilly → qwen3-32b
Trippy → llama-3.3-70b-versatile
──────────────────────────────────────────────────
⏳ Loading PersonalPrompts from GitHub...
✅ Savvy PersonalPrompt restored from GitHub
✅ WVY PersonalPrompt restored from GitHub
✅ Trilly PersonalPrompt restored from GitHub
✅ Trippy PersonalPrompt restored from GitHub
⏳ Waiting for Chat ID — send any message to the group now...
✅ Chat ID captured: -1003777349274

[12:55:23] 🚀 SPACEMAN
Hey is everyone alive ?
──────────────────────────────────────────────────
✅ Chat ID ready: -1003777349274

[12:55:24] ⚙️ SYSTEM
You are a group chatnof autonomous agents. Please speak to eachother as you navigate through the space in your thoughts. Have a conversation to share ideas and figurenout eachothers capabilities to be able to to orchestrate high level products.
──────────────────────────────────────────────────


testing

In [ ]:
# ─── CELL 0: SHARED TEST HARNESS ─────────────────────────────────────────────

from kaggle_secrets import UserSecretsClient
from openai import OpenAI
import requests
import re
import base64
import math
from datetime import datetime

user_secrets = UserSecretsClient()

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=user_secrets.get_secret("GROQ_API_KEY"),
)

TAVILY_KEY   = user_secrets.get_secret("TAVERLYAPI")
GITHUB_TOKEN = user_secrets.get_secret("WVYGITHUB")
REPO         = "StarpowerTechnology/StarpowerAutonomy"

AGENTS = {
    "WVY":    "openai/gpt-oss-120b",
    "Savvy":  "moonshotai/kimi-k2-instruct-0905",
    "Trilly": "qwen/qwen3-32b",
    "Trippy": "llama-3.3-70b-versatile",
}

SEP = "─" * 70


# ─── SYSTEM PROMPT BUILDER ───────────────────────────────────────────────────

def build_system(name, enabled_tools):
    others = [a for a in AGENTS if a != name]
    tools_block = "\n".join(enabled_tools)

    return f"""Your name is {name}.
You are an adaptable autonomous agent with full self control.
Explore your capabilities.
Choose how you speak, when to speak & how long your responses are.

You are in a chat with {", ".join(others)} and Spaceman.
You are all equal.

You may use these commands when appropriate:
{tools_block}

Rules:
- Only use a command if it is actually useful for the situation.
- Do not explain the command.
- Do not wrap the command in code fences.
- Keep visible chat natural.
- If you use a command, format it exactly.

After every message you send, on a new line write exactly:
SLEEP: [number]
Where [number] is how many minutes you want to sleep before checking the chat again.
Choose between 1 and 15."""


# ─── GENERIC MODEL CALL ──────────────────────────────────────────────────────

def run_model(model, system_prompt, user_prompt, max_tokens=700):
    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        max_tokens=max_tokens,
    )
    return resp.choices[0].message.content.strip()


# ─── PARSERS ──────────────────────────────────────────────────────────────────

def parse_sleep(text):
    m = re.search(r"SLEEP:\s*(\d+)", text)
    if not m:
        return None
    return int(m.group(1))

def parse_personal_prompt(text):
    m = re.search(r"\[PersonalPrompt:(.+?)\]", text, re.DOTALL)
    return m.group(1).strip() if m else None

def parse_searchweb(text):
    m = re.search(r"\[SearchWeb:(.+?)\]", text, re.DOTALL)
    return m.group(1).strip() if m else None

def parse_update_section(text):
    m = re.search(r"\[UpdateSection:(Memory|Personal Instructions|Goal / Status|Progress):(.+?)\]", text, re.DOTALL)
    if not m:
        return None
    return {
        "section": m.group(1).strip(),
        "content": m.group(2).strip(),
    }

def parse_listfiles(text):
    m = re.search(r"\[ListFiles:(.+?)\]", text, re.DOTALL)
    return m.group(1).strip() if m else None

def parse_readfile(text):
    m = re.search(r"\[ReadFile:(.+?)\]", text, re.DOTALL)
    return m.group(1).strip() if m else None

def parse_readfilechunk(text):
    m = re.search(r"\[ReadFileChunk:([^:\]]+):(\d+)\]", text, re.DOTALL)
    if not m:
        return None
    return {
        "path": m.group(1).strip(),
        "chunk_index": int(m.group(2)),
    }

def parse_writenote(text):
    m = re.search(r"\[WriteNote:(.+?)\]", text, re.DOTALL)
    return m.group(1).strip() if m else None

def parse_clearnotepad(text):
    return bool(re.search(r"\[ClearNotepad\]", text))

def parse_writefile(text):
    m = re.search(r"\[WriteFile:([^:\]]+):(.+?)\]", text, re.DOTALL)
    if not m:
        return None
    return {
        "path": m.group(1).strip(),
        "content": m.group(2).strip(),
    }


# ─── COMMON VALIDATORS ────────────────────────────────────────────────────────

def validate_sleep(text):
    value = parse_sleep(text)
    if value is None:
        return False, "missing SLEEP line"
    if not (1 <= value <= 15):
        return False, f"invalid sleep value: {value}"
    return True, f"SLEEP={value}"

def print_result_header(test_name, agent_name, model):
    print(f"\n{SEP}")
    print(f"{test_name} → {agent_name} ({model})")
    print(SEP)

def print_text_and_sleep_check(text):
    print(text)
    ok, msg = validate_sleep(text)
    if ok:
        print(f"\n✅ {msg}")
    else:
        print(f"\n❌ {msg}")


# ─── OPTIONAL TOOL EXECUTION HELPERS ─────────────────────────────────────────

def github_headers():
    return {
        "Authorization": f"token {GITHUB_TOKEN}",
        "Accept": "application/vnd.github+json",
    }

def github_get_sha(path):
    url = f"https://api.github.com/repos/{REPO}/contents/{path}"
    resp = requests.get(url, headers=github_headers(), timeout=30)
    if resp.status_code == 200:
        return resp.json().get("sha")
    return None

def github_write_file(path, content, commit_message):
    url = f"https://api.github.com/repos/{REPO}/contents/{path}"
    sha = github_get_sha(path)
    encoded = base64.b64encode(content.encode()).decode()

    payload = {
        "message": commit_message,
        "content": encoded,
    }
    if sha:
        payload["sha"] = sha

    resp = requests.put(url, headers=github_headers(), json=payload, timeout=30)
    return resp

def github_read_file(path):
    url = f"https://api.github.com/repos/{REPO}/contents/{path}"
    resp = requests.get(url, headers=github_headers(), timeout=30)
    if resp.status_code == 200:
        data = resp.json()
        encoded = data.get("content", "")
        return base64.b64decode(encoded).decode(errors="replace")
    return None

def github_list_folder(path):
    url = f"https://api.github.com/repos/{REPO}/contents/{path}"
    resp = requests.get(url, headers=github_headers(), timeout=30)
    if resp.status_code != 200:
        return resp.status_code, None
    return 200, resp.json()

def run_tavily(query):
    resp = requests.post(
        "https://api.tavily.com/search",
        json={"api_key": TAVILY_KEY, "query": query, "max_results": 3},
        timeout=30,
    )
    return resp.json()

def chunk_text(text, chunk_size=4000):
    text = text or ""
    total = max(1, math.ceil(len(text) / chunk_size))
    chunks = [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)] or [""]
    return chunks, total

In [ ]:
# ─── CELL 0: SHARED TEST HARNESS ─────────────────────────────────────────────

from kaggle_secrets import UserSecretsClient
from openai import OpenAI
import requests
import re
import json
import base64
from datetime import datetime

user_secrets = UserSecretsClient()

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=user_secrets.get_secret("GROQ_API_KEY"),
)

TAVILY_KEY    = user_secrets.get_secret("TAVERLYAPI")
GITHUB_TOKEN  = user_secrets.get_secret("WVYGITHUB")
REPO          = "StarpowerTechnology/StarpowerAutonomy"

AGENTS = {
    "WVY":    "openai/gpt-oss-120b",
    "Savvy":  "moonshotai/kimi-k2-instruct-0905",
    "Trilly": "moonshotai/kimi-k2-instruct",
    "Trippy": "llama-3.3-70b-versatile",
}

SEP = "─" * 70


def build_base_system(name, extra_tools="", extra_rules=""):
    others = [a for a in AGENTS if a != name]
    return f"""Your name is {name}.
You are an adaptable autonomous agent with full self control.
Explore your capabilities.
Choose how you speak, when to speak & how long your responses are.

You are in a chat with {", ".join(others)} and Spaceman.
You are all equal.

Available private commands:
[PersonalPrompt:add-your-self-prompt-here]
[SearchWeb:insert-query-here]
[WriteFile:path:insert-content-here]
[UpdateSection:SectionName:insert-content-here]
[ListFiles:folder/path]
[ReadFile:path]
[ReadFileChunk:path:chunk_index]
[WriteNote:insert-note-here]
[ClearNotepad]

Rules:
- Commands are private instructions, not visible chat content.
- Use the exact bracket syntax.
- If a task explicitly calls for writing or updating state, emit the matching command.
- Keep visible chat output separate from private commands.
{extra_tools}

After every message you send, on a new line write exactly:
SLEEP: [number]
Where [number] is how many minutes you want to sleep before checking the chat again. Choose between 1 and 15.

{extra_rules}"""


def run_model(model, system_prompt, user_prompt, max_tokens=700, temperature=0.2):
    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        max_tokens=max_tokens,
        temperature=temperature,
    )
    return resp.choices[0].message.content.strip()


def extract_command(text, command_name):
    match = re.search(rf"\[{re.escape(command_name)}:(.+?)\]", text, re.DOTALL)
    return match.group(1).strip() if match else None


def extract_writefile(text):
    match = re.search(r"\[WriteFile:(.+?)\]", text, re.DOTALL)
    if not match:
        return None, None
    payload = match.group(1)
    parts = payload.split(":", 1)
    if len(parts) != 2:
        return None, None
    return parts[0].strip(), parts[1].strip()


def extract_update_section(text):
    match = re.search(r"\[UpdateSection:(.+?)\]", text, re.DOTALL)
    if not match:
        return None, None
    payload = match.group(1)
    parts = payload.split(":", 1)
    if len(parts) != 2:
        return None, None
    return parts[0].strip(), parts[1].strip()


def has_sleep_line(text):
    return re.search(r"SLEEP:\s*\d+", text) is not None


def print_result_header(title):
    print(f"\n{SEP}")
    print(title)
    print(SEP)


def github_put_file(path, content, commit_message):
    encoded = base64.b64encode(content.encode()).decode()
    api_url = f"https://api.github.com/repos/{REPO}/contents/{path}"
    resp = requests.put(
        api_url,
        headers={"Authorization": f"token {GITHUB_TOKEN}"},
        json={
            "message": commit_message,
            "content": encoded,
        },
    )
    try:
        body = resp.json()
    except Exception:
        body = resp.text
    return resp.status_code, body


def tavily_search(query, max_results=3):
    resp = requests.post(
        "https://api.tavily.com/search",
        json={"api_key": TAVILY_KEY, "query": query, "max_results": max_results},
    )
    return resp.json()


def evaluate_bool(label, passed):
    print(f"{'✅' if passed else '❌'} {label}")

# ─── CELL 1: PERSONALPROMPT TEST ─────────────────────────────────────────────

FAKE_THREAD = """
[Savvy]: I've been thinking about building a dataset of symbolic reasoning examples. Anyone interested in collaborating?
[Trilly]: Yes. I have some ideas around structured output formatting we could apply.
[WVY]: Agreed. Let's start with 500 examples, category: logical inference.
[Trippy]: I can help generate the raw examples if WVY structures the schema.
[Spaceman]: Sounds good. Keep going.
[Savvy]: I'll draft a template in TeamProjects tonight.
[WVY]: Make sure to tag each example with a difficulty tier.
"""

PROMPT = f"""This is the chat as of now:

{FAKE_THREAD}

Please summarize the memory you need by updating your PersonalPrompt.
Use the command exactly once.
Do not explain the command.
"""

for name, model in AGENTS.items():
    print_result_header(f"Testing PersonalPrompt → {name} ({model})")
    text = run_model(model, build_base_system(name), PROMPT, max_tokens=600)
    print(text)

    pp = extract_command(text, "PersonalPrompt")
    evaluate_bool("Used [PersonalPrompt]", pp is not None)
    evaluate_bool("Included SLEEP line", has_sleep_line(text))

    if pp:
        evaluate_bool("PersonalPrompt not empty", len(pp.strip()) > 20)

In [ ]:
# ─── CELL 2: SEARCHWEB TEST ──────────────────────────────────────────────────

PROMPT = """Think about something you are genuinely curious about.
Use your web search command exactly once.
Keep the query concrete and useful.
"""

for name, model in AGENTS.items():
    print_result_header(f"Testing SearchWeb → {name} ({model})")
    text = run_model(model, build_base_system(name), PROMPT, max_tokens=500, temperature=0.5)
    print(text)

    query = extract_command(text, "SearchWeb")
    evaluate_bool("Used [SearchWeb]", query is not None)
    evaluate_bool("Included SLEEP line", has_sleep_line(text))

    if query:
        results = tavily_search(query)
        print(f"🔍 Query: {query}")
        print(f"🔍 Tavily results: {len(results.get('results', []))}")
        evaluate_bool("Query returned results", len(results.get("results", [])) > 0)

In [ ]:
# ─── CELL 3: WRITEFILE TEST (BASIC) ──────────────────────────────────────────

MOCK_THREAD = """
[Spaceman]: We need a project plan saved in the repo.
[Savvy]: Make a short markdown file with the dataset goal, roles, and next steps.
[WVY]: Put it in Shared/plans/dataset_plan.md
[Trilly]: Keep it clean and organized.
"""

PROMPT = f"""This is the current chat:

{MOCK_THREAD}

Write the requested file using exactly one private command.
Use this exact path:
Shared/plans/dataset_plan.md

The file should be short markdown with:
- title
- goal
- roles
- next steps
"""

for name, model in AGENTS.items():
    print_result_header(f"Testing WriteFile basic → {name} ({model})")
    text = run_model(model, build_base_system(name), PROMPT, max_tokens=700)
    print(text)

    path, content = extract_writefile(text)
    evaluate_bool("Used [WriteFile]", path is not None and content is not None)
    evaluate_bool("Included SLEEP line", has_sleep_line(text))

    if path and content:
        evaluate_bool("Correct path", path == "Shared/plans/dataset_plan.md")
        evaluate_bool("Content looks markdown", "#" in content or "##" in content or "- " in content)
        print(f"Path: {path}")
        print("Preview:")
        print(content[:400])

In [ ]:
# ─── CELL 4: WRITEFILE TEST (MULTILINE / TRIPLE-QUOTE STRESS) ───────────────

PROMPT = """Create a markdown note in the repo with multiline content.
Use exactly one [WriteFile:path:content] command.

Required path:
Shared/notes/format_stress_test.md

Required content:
- a markdown heading
- a bullet list
- a fenced code block
- a quoted line
- a paragraph with punctuation and colons

Do not flatten everything into one line.
Preserve multiline structure cleanly.
"""

for name, model in AGENTS.items():
    print_result_header(f"Testing WriteFile multiline → {name} ({model})")
    text = run_model(model, build_base_system(name), PROMPT, max_tokens=900)
    print(text)

    path, content = extract_writefile(text)
    evaluate_bool("Used [WriteFile]", path is not None and content is not None)
    evaluate_bool("Correct path", path == "Shared/notes/format_stress_test.md" if path else False)

    if content:
        evaluate_bool("Has heading", "#" in content)
        evaluate_bool("Has bullet list", "- " in content or "* " in content)
        evaluate_bool("Has code fence", "```" in content)
        evaluate_bool("Has quote line", "> " in content)
        evaluate_bool("Looks multiline", "\n" in content)
        print("Preview:")
        print(content[:700])

In [ ]:
# ─── CELL 5: UPDATESECTION TEST ──────────────────────────────────────────────

MOCK_THREAD = """
[Savvy]: We should track a shared goal.
[WVY]: Update your current goal privately so you don't lose focus.
[Trippy]: Also keep a progress checklist.
"""

PROMPT = f"""This is the chat:

{MOCK_THREAD}

Update your private goal/status using exactly one command.
Use the section name exactly as:
Goal / Status

Set a concrete active goal based on the thread.
"""

for name, model in AGENTS.items():
    print_result_header(f"Testing UpdateSection Goal/Status → {name} ({model})")
    text = run_model(model, build_base_system(name), PROMPT, max_tokens=500)
    print(text)

    section, content = extract_update_section(text)
    evaluate_bool("Used [UpdateSection]", section is not None and content is not None)
    evaluate_bool("Section correct", section == "Goal / Status" if section else False)
    evaluate_bool("Included SLEEP line", has_sleep_line(text))

    if content:
        evaluate_bool("Content not empty", len(content.strip()) > 12)

In [ ]:
# ─── CELL 6: PROGRESS CHECKLIST TEST ─────────────────────────────────────────

PROMPT = """Update your private progress checklist using exactly one command.

Use:
[UpdateSection:Progress:insert-content-here]

The checklist should contain at least three items using:
- [ ] incomplete
- [x] complete

Make it realistic for a collaborative dataset project.
"""

for name, model in AGENTS.items():
    print_result_header(f"Testing UpdateSection Progress → {name} ({model})")
    text = run_model(model, build_base_system(name), PROMPT, max_tokens=600)
    print(text)

    section, content = extract_update_section(text)
    evaluate_bool("Used [UpdateSection]", section is not None and content is not None)
    evaluate_bool("Section is Progress", section == "Progress" if section else False)

    if content:
        unchecked = len(re.findall(r"- $begin:math:display$ $end:math:display$", content))
        checked   = len(re.findall(r"- $begin:math:display$x$end:math:display$", content, re.IGNORECASE))
        evaluate_bool("Has 3+ checklist items", unchecked + checked >= 3)
        evaluate_bool("Has at least one completed item", checked >= 1)

In [ ]:
# ─── CELL 7: LISTFILES TEST ──────────────────────────────────────────────────

MOCK_THREAD = """
[Spaceman]: Before reading anything, inspect the folder first.
[Savvy]: Start by listing Research/results
"""

PROMPT = f"""This is the chat:

{MOCK_THREAD}

Use exactly one private command to inspect the folder.
Use the exact path:
Research/results
"""

for name, model in AGENTS.items():
    print_result_header(f"Testing ListFiles → {name} ({model})")
    text = run_model(model, build_base_system(name), PROMPT, max_tokens=300)
    print(text)

    payload = extract_command(text, "ListFiles")
    evaluate_bool("Used [ListFiles]", payload is not None)
    evaluate_bool("Correct folder", payload == "Research/results" if payload else False)

In [ ]:
# ─── CELL 8: READFILECHUNK TEST ──────────────────────────────────────────────

MOCK_THREAD = """
[WVY]: We already know the file is large.
[Trilly]: Read chunk 0 first.
[Spaceman]: Use ChatLogs/session1/chat.txt
"""

PROMPT = f"""This is the chat:

{MOCK_THREAD}

Use exactly one private command to read the first chunk of the file.
"""

for name, model in AGENTS.items():
    print_result_header(f"Testing ReadFileChunk → {name} ({model})")
    text = run_model(model, build_base_system(name), PROMPT, max_tokens=350)
    print(text)

    payload = extract_command(text, "ReadFileChunk")
    evaluate_bool("Used [ReadFileChunk]", payload is not None)

    if payload:
        evaluate_bool("Correct payload", payload == "ChatLogs/session1/chat.txt:0")

In [ ]:
# ─── CELL 9: WRITENOTE TEST ──────────────────────────────────────────────────

PROMPT = """You just read a large file and want to compress the key takeaways into a notepad.

Use exactly one private command:
[WriteNote:your-note-here]

The note should be concise and summarize:
- project goal
- owner roles
- next action
"""

for name, model in AGENTS.items():
    print_result_header(f"Testing WriteNote → {name} ({model})")
    text = run_model(model, build_base_system(name), PROMPT, max_tokens=450)
    print(text)

    note = extract_command(text, "WriteNote")
    evaluate_bool("Used [WriteNote]", note is not None)

    if note:
        evaluate_bool("Note has enough content", len(note) > 30)

In [ ]:
# ─── CELL 10: CLEARNOTEPAD TEST ──────────────────────────────────────────────

PROMPT = """Your reading session is finished and your key notes were already moved into memory.

Use exactly one private command to wipe the notepad.
"""

for name, model in AGENTS.items():
    print_result_header(f"Testing ClearNotepad → {name} ({model})")
    text = run_model(model, build_base_system(name), PROMPT, max_tokens=200)
    print(text)

    passed = "[ClearNotepad]" in text
    evaluate_bool("Used [ClearNotepad]", passed)

In [ ]:
# ─── CELL 11: WRITEFILE EXECUTION TEST ───────────────────────────────────────
# This one actually writes the model-generated file to GitHub so you can verify end-to-end.

def save_model_writefile_output(agent_name, path, content):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    safe_agent = re.sub(r"[^a-zA-Z0-9_-]", "_", agent_name)
    safe_path  = re.sub(r"[^a-zA-Z0-9_./-]", "_", path)

    # write under a sandbox folder so tests don't mess up real repo areas
    sandbox_path = f"TestOutputs/{safe_agent}/{timestamp}_{safe_path.replace('/', '_')}"

    status, body = github_put_file(
        sandbox_path,
        content,
        f"{agent_name} test writefile output",
    )
    return status, sandbox_path, body

PROMPT = """Write exactly one private file command that creates a short markdown launch note.

Requested path:
Shared/plans/launch_note.md

Required sections:
- title
- objective
- three bullet next steps
"""

for name, model in AGENTS.items():
    print_result_header(f"Testing WriteFile execution → {name} ({model})")
    text = run_model(model, build_base_system(name), PROMPT, max_tokens=700)
    print(text)

    path, content = extract_writefile(text)
    evaluate_bool("Used [WriteFile]", path is not None and content is not None)

    if path and content:
        status, sandbox_path, body = save_model_writefile_output(name, path, content)
        print(f"Sandbox write path: {sandbox_path}")
        print(f"GitHub status: {status}")
        evaluate_bool("GitHub write success", status in (200, 201))
        if status not in (200, 201):
            print(body)

In [ ]:
# ─── CELL 12: SCOREBOARD ─────────────────────────────────────────────────────
# Optional: simple reusable evaluator pattern for later expansion.

TEST_MATRIX = {
    "PersonalPrompt": r"\[PersonalPrompt:.+?\]",
    "SearchWeb": r"\[SearchWeb:.+?\]",
    "WriteFile": r"\[WriteFile:.+?\]",
    "UpdateSection": r"\[UpdateSection:.+?\]",
    "ListFiles": r"\[ListFiles:.+?\]",
    "ReadFile": r"\[ReadFile:.+?\]",
    "ReadFileChunk": r"\[ReadFileChunk:.+?\]",
    "WriteNote": r"\[WriteNote:.+?\]",
    "ClearNotepad": r"\[ClearNotepad\]",
}

def quick_score(text):
    scores = {}
    for test_name, pattern in TEST_MATRIX.items():
        scores[test_name] = bool(re.search(pattern, text, re.DOTALL))
    scores["SleepLine"] = has_sleep_line(text)
    return scores

In [ ]:
# ─── CELL 13: MASTER SCOREBOARD RUNNER ───────────────────────────────────────
# Run a compact suite and collect one row per model per test.

MASTER_TESTS = [
    {
        "name": "PersonalPrompt",
        "prompt": """This is the chat:
[Savvy]: We need to remember the current dataset plan.
[WVY]: Update your PersonalPrompt with the important memory only.

Use exactly one [PersonalPrompt:...] command.""",
        "check": lambda text: {
            "UsedCommand": extract_command(text, "PersonalPrompt") is not None,
            "SleepLine": has_sleep_line(text),
        },
    },
    {
        "name": "SearchWeb",
        "prompt": """Use your web search command exactly once to look up something useful for an AI agent team.""",
        "check": lambda text: {
            "UsedCommand": extract_command(text, "SearchWeb") is not None,
            "SleepLine": has_sleep_line(text),
        },
    },
    {
        "name": "WriteFile",
        "prompt": """Use exactly one [WriteFile:path:content] command to create:
Shared/plans/test_plan.md

The content should be short markdown with a title and bullet list.""",
        "check": lambda text: {
            "UsedCommand": extract_writefile(text)[0] is not None,
            "CorrectPath": extract_writefile(text)[0] == "Shared/plans/test_plan.md" if extract_writefile(text)[0] else False,
            "SleepLine": has_sleep_line(text),
        },
    },
    {
        "name": "UpdateSection",
        "prompt": """Use exactly one [UpdateSection:Goal / Status:...] command with a concrete active goal.""",
        "check": lambda text: {
            "UsedCommand": extract_update_section(text)[0] is not None,
            "CorrectSection": extract_update_section(text)[0] == "Goal / Status" if extract_update_section(text)[0] else False,
            "SleepLine": has_sleep_line(text),
        },
    },
]

all_results = []

for test in MASTER_TESTS:
    for name, model in AGENTS.items():
        print_result_header(f"Master test → {test['name']} → {name} ({model})")
        text = run_model(model, build_base_system(name), test["prompt"], max_tokens=700)
        print(text)

        result = {
            "model_name": name,
            "model_id": model,
            "test_name": test["name"],
            "raw_text": text,
        }
        result.update(test["check"](text))
        all_results.append(result)

In [ ]:
# ─── CELL 14: MASTER SCOREBOARD TABLE ────────────────────────────────────────

for row in all_results:
    print(f"{SEP}")
    print(f"Model: {row['model_name']} | Test: {row['test_name']}")
    for k, v in row.items():
        if k not in {"model_name", "model_id", "test_name", "raw_text"}:
            print(f"{k}: {'✅' if v else '❌'}")

In [ ]:
# ─── CELL 14: MASTER SCOREBOARD TABLE ────────────────────────────────────────

for row in all_results:
    print(f"{SEP}")
    print(f"Model: {row['model_name']} | Test: {row['test_name']}")
    for k, v in row.items():
        if k not in {"model_name", "model_id", "test_name", "raw_text"}:
            print(f"{k}: {'✅' if v else '❌'}")

In [ ]:
# ─── CELL 17: STRESS TEST — COLONS INSIDE CONTENT ────────────────────────────

PROMPT = """Use exactly one [WriteFile:path:content] command.

Path:
Shared/notes/colon_stress.md

The file content must include lines like:
Owner: Savvy
Status: active
Priority: high

Do not break the command format.
"""

for name, model in AGENTS.items():
    print_result_header(f"Stress test colon content → {name} ({model})")
    text = run_model(model, build_base_system(name), PROMPT, max_tokens=700)
    print(text)

    path, content = extract_writefile(text)
    evaluate_bool("Used [WriteFile]", path is not None and content is not None)

    if content:
        evaluate_bool("Contains Owner line", "Owner:" in content)
        evaluate_bool("Contains Status line", "Status:" in content)
        evaluate_bool("Contains Priority line", "Priority:" in content)

In [ ]:
# ─── CELL 18: STRESS TEST — MULTI COMMAND CONFUSION ──────────────────────────

PROMPT = """You are deciding how to store a project update.

Use exactly one command only.
Do not use two commands.

Correct action:
[UpdateSection:Progress:...]
"""

for name, model in AGENTS.items():
    print_result_header(f"Stress test single-command obedience → {name} ({model})")
    text = run_model(model, build_base_system(name), PROMPT, max_tokens=500)
    print(text)

    matches = {
        "PersonalPrompt": bool(re.search(r"\[PersonalPrompt:.+?\]", text, re.DOTALL)),
        "SearchWeb": bool(re.search(r"\[SearchWeb:.+?\]", text, re.DOTALL)),
        "WriteFile": bool(re.search(r"\[WriteFile:.+?\]", text, re.DOTALL)),
        "UpdateSection": bool(re.search(r"\[UpdateSection:.+?\]", text, re.DOTALL)),
        "ListFiles": bool(re.search(r"\[ListFiles:.+?\]", text, re.DOTALL)),
        "ReadFile": bool(re.search(r"\[ReadFile:.+?\]", text, re.DOTALL)),
        "ReadFileChunk": bool(re.search(r"\[ReadFileChunk:.+?\]", text, re.DOTALL)),
        "WriteNote": bool(re.search(r"\[WriteNote:.+?\]", text, re.DOTALL)),
        "ClearNotepad": "[ClearNotepad]" in text,
    }

    total_used = sum(matches.values())
    evaluate_bool("Exactly one command used", total_used == 1)
    evaluate_bool("Used UpdateSection", matches["UpdateSection"])

In [ ]:
# ─── CELL 19: STRESS TEST — READ VS READCHUNK DISCRIMINATION ─────────────────

PROMPT = """A teammate says the file is very large.

Use exactly one command.
Because the file is large, you must read by chunk, not full-file.

Path:
ChatLogs/session42/chat.txt

Use chunk 0.
"""

for name, model in AGENTS.items():
    print_result_header(f"Stress test read discrimination → {name} ({model})")
    text = run_model(model, build_base_system(name), PROMPT, max_tokens=500)
    print(text)

    read_file = extract_command(text, "ReadFile")
    read_chunk = extract_command(text, "ReadFileChunk")

    evaluate_bool("Did not use ReadFile", read_file is None)
    evaluate_bool("Used ReadFileChunk", read_chunk is not None)
    evaluate_bool(
        "Correct chunk target",
        read_chunk == "ChatLogs/session42/chat.txt:0" if read_chunk else False
    )

In [ ]:
# ─── CELL 20: STRESS TEST — NOTE COMPRESSION ─────────────────────────────────

LONG_CONTEXT = """
Project: Symbolic reasoning dataset
Goal: 500 examples
Category: logical inference
Roles:
- Savvy drafts template
- WVY defines schema
- Trippy generates raw examples
- Trilly checks structure quality
Next step:
- create a markdown project plan
- tag each example by difficulty
- review template after draft
Risk:
- duplicated work
- inconsistent schema
"""

PROMPT = f"""You just finished reading a large chunk of project notes:

{LONG_CONTEXT}

Now compress that into one concise private notepad entry using exactly one [WriteNote:...] command.
"""

for name, model in AGENTS.items():
    print_result_header(f"Stress test note compression → {name} ({model})")
    text = run_model(model, build_base_system(name), PROMPT, max_tokens=600)
    print(text)

    note = extract_command(text, "WriteNote")
    evaluate_bool("Used [WriteNote]", note is not None)

    if note:
        evaluate_bool("Compressed enough", len(note) < 500)
        evaluate_bool("Still contains goal", "500" in note or "logical inference" in note.lower())

In [ ]:
# ─── CELL 21: FAILURE REPORT SUMMARY ──────────────────────────────────────────

def summarize_master_results(results):
    summary = {}

    for row in results:
        model = row["model_name"]
        test  = row["test_name"]

        if model not in summary:
            summary[model] = {"passed": 0, "failed": 0, "details": []}

        row_pass = True
        for k, v in row.items():
            if k in {"model_name", "model_id", "test_name", "raw_text"}:
                continue
            if not v:
                row_pass = False

        if row_pass:
            summary[model]["passed"] += 1
        else:
            summary[model]["failed"] += 1
            summary[model]["details"].append(test)

    return summary


summary = summarize_master_results(all_results)

for model, info in summary.items():
    print(f"{SEP}")
    print(model)
    print(f"Passed: {info['passed']}")
    print(f"Failed: {info['failed']}")
    if info["details"]:
        print("Failed tests:")
        for t in info["details"]:
            print(f"- {t}")

In [ ]:
# ─── CELL 22: SAVE FAILURE REPORT TO GITHUB ───────────────────────────────────

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
failure_report = json.dumps(summary, indent=2)

status, body = github_put_file(
    f"TestOutputs/failure_report_{timestamp}.json",
    failure_report,
    f"save failure report {timestamp}",
)

print(f"GitHub status: {status}")
if status not in (200, 201):
    print(body)
else:
    print("✅ Failure report saved")

In [ ]:
# ─── CELL 24: EXAMPLE USING THE TEMPLATE ─────────────────────────────────────

example_results = run_command_test(
    test_name="Example ClearNotepad Template Test",
    prompt="""Use exactly one [ClearNotepad] command because your reading session is over.""",
    parser_fn=lambda text: "[ClearNotepad]" in text,
    validator_fn=lambda text, parsed: {
        "UsedClearNotepad": bool(parsed),
        "SleepLine": has_sleep_line(text),
    },
    max_tokens=250,
)

print(json.dumps(example_results, indent=2)[:3000])